# DO2 Multiclass Status Prediction — Java-Condition ML Features — Safety Review Decision Revision

This revision predicts `I`, `MR`, `N`, or `V` from entry-time bill properties
and point-in-time historical facts. Calendar month/year and monthly status
percentages are forbidden model inputs. Feature selection is based on temporal
stability, extreme delay is capped, the stable pattern key excludes temporary
source/delay combinations, and the deployment model is refit through June 2025
before the untouched July–December 2025 test is evaluated.


This version keeps `pattern_*` columns for diagnostics only. They are explicitly forbidden from CatBoost model inputs.


Operational output is now `I`, `MR`, `N`, `V`, or `REVIEW`; uncertain rows never fall back to `V`.

## 1. Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT_HINT = Path(
    "/home/ruthikreddy/Desktop/do2/do2_status_prediction"
)
project_candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent, PROJECT_ROOT_HINT]
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / "pyproject.toml").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the do2_status_prediction project.")

DO2_CHECKS_PATH = PROJECT_ROOT / "src" / "do2_blocking_checks"
if not DO2_CHECKS_PATH.exists():
    raise RuntimeError(f"Stage 2 feature package not found: {DO2_CHECKS_PATH}")
if str(DO2_CHECKS_PATH) not in sys.path:
    sys.path.insert(0, str(DO2_CHECKS_PATH))


print("DO2 Stage 2 feature package available:", DO2_CHECKS_PATH)


DO2 Stage 2 feature package available: /home/ruthikreddy/Desktop/do2/do2_status_prediction/src/do2_blocking_checks


In [2]:
import gc
import importlib.metadata
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sqlalchemy import text

cwd = Path.cwd().resolve()
project_candidates = [cwd, cwd.parent, cwd / "do2_status_prediction"]
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / "pyproject.toml").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Run this notebook from the do2_status_prediction project root or its src directory."
    )

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from do2_model.config import Settings
from do2_model.database import get_engine

settings = Settings()
if not settings.database_url:
    raise RuntimeError(f"DATABASE_URL is missing from {PROJECT_ROOT / '.env'}")

engine = get_engine(settings)

print(f"Project: {PROJECT_ROOT}")
print(f"Database schema: {settings.db_schema}")
print("DATABASE_URL loaded from .env (value hidden).")


Project: /home/ruthikreddy/Desktop/do2/do2_status_prediction
Database schema: public
DATABASE_URL loaded from .env (value hidden).


In [3]:
# Verify the same SQLAlchemy engine used by the training pipeline.
with engine.connect() as connection:
    database_name, schema_name = connection.execute(
        text("SELECT current_database(), current_schema()")
    ).one()

if schema_name != settings.db_schema:
    raise RuntimeError(
        f"Configured schema {settings.db_schema!r} does not match "
        f"the connection schema {schema_name!r}."
    )

print("Database:", database_name)
print("Schema:", schema_name)
print("PostgreSQL connection verified; credentials remain hidden.")


Database: do2
Schema: public
PostgreSQL connection verified; credentials remain hidden.


## 2. Parameters and temporal split policy

In [4]:
# PostgreSQL source. Every split is selected from the same source table.
TRAIN_TABLE = "do2"
SOURCE_TABLE = "do2"

# Use the timestamp at which a record became available to the model.
SPLIT_DATE_COLUMN = "created_at"

TARGET_MODE = "reduced_lifecycle_risk"

CORE_OUTCOME_STATUSES = ("I", "MR", "N", "V")

SNAPSHOT_STATUSES = ("C", "I", "MR", "N", "P", "V")

LOAD_STATUSES = SNAPSHOT_STATUSES

if TARGET_MODE == "reduced_lifecycle_risk":

    VALID_STATUSES = CORE_OUTCOME_STATUSES

elif TARGET_MODE == "six_status_snapshot":

    VALID_STATUSES = SNAPSHOT_STATUSES

else:

    raise ValueError(f"Unknown TARGET_MODE: {TARGET_MODE}")

VALID_DO2_GROUPS = ("IA", "IB", "IIA", "IIB", "ID","IC")

TRAIN_START = "2022-01-01"
TRAIN_END = "2025-01-01"
VALIDATION_START = "2025-01-01"
VALIDATION_END = "2025-07-01"
TEST_START = "2025-07-01"
TEST_END = "2026-01-01"

# Keep every eligible non-V training row and a deterministic V sample.
# Reduce this value if memory is limited.
MAX_TRAIN_V_ROWS = 250_000

# Validation and test remain natural-distribution samples.
# Use None only when the machine can hold the complete split.
MAX_VALIDATION_ROWS = 250_000
MAX_TEST_ROWS = 250_000

# Cache expensive enriched query results. Set REFRESH_DATA_CACHE=True only when
# source data or enrichment SQL changes.
USE_DATA_CACHE = True
REFRESH_DATA_CACHE = False
DATA_CACHE_VERSION = "time_safe_v7_java_n_features_no_pattern_model"
DATA_CACHE_DIR = PROJECT_ROOT / "data" / "model_cache"

# This is allowed only for known bad historical training labels. Validation and
# test never use the target status to remove rows.
DROP_INCOMPLETE_V_TRAINING_LABELS = True

# If a row's created_at cannot be parsed (NULL, malformed text, or a
# PostgreSQL "infinity"/"-infinity" sentinel timestamp) it cannot be
# time-split safely. These rows are excluded before splitting and the
# excluded count is reported, instead of silently becoming NaT and
# failing the split_validation check later.
DROP_UNPARSEABLE_CREATED_AT_ROWS = True

# Softly correct the deliberately sampled training distribution toward the
# natural class distribution in the training period.
# Swept automatically in section 8 against validation macro-F1 instead of
# being fixed. This is only the fallback/default if the sweep is skipped.
PRIOR_CORRECTION_POWER = 0.5
PRIOR_CORRECTION_POWER_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)


# Optional, bounded feature-group ablation. It is valuable for proving that

# each rule family adds validation-month performance, but it trains extra models.

RUN_FEATURE_GROUP_ABLATION = True

ABLATION_ITERATIONS = 350

# The three development folds are evaluated in chronological order.
# These dates select rows only; they are never supplied as model features.
STABILITY_FOLDS = (
    ("2024_H1", "2024-01-01", "2024-07-01"),
    ("2024_H2", "2024-07-01", "2025-01-01"),
    ("2025_H1", "2025-01-01", "2025-07-01"),
)
MIN_STABILITY_FOLD_ROWS = 1_000
REFIT_WITH_VALIDATION = True

# Class-specific probability cutoffs are learned from validation only. A
# non-V argmax below its class cutoff falls back to V.
CLASS_THRESHOLD_GRID = (0.0, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80)
CLASS_THRESHOLD_PASSES = 2

# Safety policy: uncertain rows must go to REVIEW, never silently fall back to V.
# The V auto-approval threshold is learned from validation labels only. At most
# this fraction of actual validation non-V rows may be auto-approved as V.
# This is an error-rate limit, not an assumption about the percentage of
# non-V bills in any future batch.
MAX_NON_V_TO_V_RATE = 0.05
MIN_AUTO_V_PRECISION = 0.98

# appendix_j is a current snapshot, not a versioned rule table. Leave its
# historical rule features disabled until a rule effective-date/version is
# available or the business confirms the rules were unchanged for 2022-2025.
USE_CURRENT_APPENDIX_J_FOR_HISTORICAL_ROWS = False


# Model settings. Change TASK_TYPE to "CPU" if GPU training is unavailable.
TASK_TYPE = "CPU"
DEVICES = "0"
ITERATIONS = 2_000
LEARNING_RATE = 0.03
DEPTH = 6
EARLY_STOPPING_ROUNDS = 150
RANDOM_SEED = 42

MODEL_DIR = PROJECT_ROOT / "models_prac_v6_stable_bill_pattern"
SAVE_ARTIFACTS = True

print("Source:", f"{settings.db_schema}.{SOURCE_TABLE}")

print("Target mode:", TARGET_MODE)

print("Model statuses:", VALID_STATUSES)

if TARGET_MODE == "six_status_snapshot":

    print("WARNING: C and P are mutable lifecycle snapshots, not stable entry outcomes.")

print("Split timestamp:", SPLIT_DATE_COLUMN)
print("Training window:", TRAIN_START, "to", TRAIN_END)
print("Validation window:", VALIDATION_START, "to", VALIDATION_END)
print("Test window:", TEST_START, "to", TEST_END)
print("Maximum sampled V training rows:", MAX_TRAIN_V_ROWS)
print("Validation row cap:", MAX_VALIDATION_ROWS)
print("Test row cap:", MAX_TEST_ROWS)
print("Use enriched-data cache:", USE_DATA_CACHE)


Source: public.do2
Target mode: reduced_lifecycle_risk
Model statuses: ('I', 'MR', 'N', 'V')
Split timestamp: created_at
Training window: 2022-01-01 to 2025-01-01
Validation window: 2025-01-01 to 2025-07-01
Test window: 2025-07-01 to 2026-01-01
Maximum sampled V training rows: 250000
Validation row cap: 250000
Test row cap: 250000
Use enriched-data cache: True


## 3. Load and enrich data

The prediction-time grain is one row per `do2.id`. Lateral history queries are
aggregated to one row per selected DO2. Status-dependent history currently uses the decision timestamp when
available and a `created_at` fallback when it is missing. Section 3
reports the fallback population so it can be replaced with a verified
point-in-time decision timestamp before production use.

The current `employee` table may still be an updated-in-place snapshot. Values
from employee rows created after the DO2 are censored, but a true employee
history table should be used later if one is available.

In [5]:
import time
import pandas as pd

from sqlalchemy import Date, DateTime, inspect, text


LOAD_STATUS_SQL = ", ".join(f"'{status}'" for status in LOAD_STATUSES)

MODEL_STATUS_SQL = ", ".join(f"'{status}'" for status in VALID_STATUSES)


# The application code treats these employee status codes as effective.
EFFECTIVE_EMPLOYEE_STATUS_CODES = ("E", "N", "R", "D", "M", "W", "T")

# Exact lists traced from the Java DO2 processor.
ABOLISHED_DESCRIPTIONS_7CPC = (
    "CILQ", "SNLQ", "FAA", "STRINC", "HCA", "FLCHCA", "ADALLC",
    "ACTGA", "COMNDO", "INSTAL", "OUTFIT", "HSINC", "HCWA",
    "CONDMT", "BOUNTY", "RLCA1", "RLCA2", "RLCA3", "RLCA4",
    "TYOJ", "TYRJ", "DAILY",
)
MUTEX_DESCRIPTIONS = (
    "TYOJ", "TYRJ", "DRDSRA", "AL", "CL", "SL", "EXTNAL", "EXTNCL",
    "EXTNSL", "TJRA", "RTYOJ", "EXTNLV", "RTYRJ", "RDAILY", "HATRIP",
    "MRA", "FRA", "SRA", "GSRA", "NRA", "ISNRA", "ISMRA", "ISFRA",
    "ISHRA", "SPLLVE", "PRPLVE", "PAL", "BAL", "ADVAL", "ACCMAL",
    "RL", "DAILY", "POSDA", "LDAILY",
)
DUPLICATE_CHECK_EXCLUSIONS = ("ATCORR", "LRA", "DA", "CEATWN", "NOK")
DUPLICATE_CEA_FAMILY = (
    "CEA", "CEAHC", "HOSSUB", "HSBTWN", "CHNOK", "CHSRNM", "CHFMNM",
)
DUPLICATE_COLUMN_AB = ("SCCIA", "MEDAL", "UID", "BIOMET")


MODEL_SOURCE_COLUMNS = {
    "id", "status", "unit_code", "fk_unit", "army_no", "fk_employee",
    "check_digit", "do2_group", "do2_type", "do2_no", "do2_item_no",
    "do2_date", "description", "from_date", "to_date", "rate", "tjra",
    "lra", "column_a", "column_b", "column_c", "ref_do2_no",
    "transcription_type", "qe", "created_at", "attachment", "delayed",
    "time_bar_sanction", "mr",
}

def quote_identifier(identifier):
    """Safely quote a PostgreSQL table or column name."""
    return '"' + identifier.replace('"', '""') + '"'


def qualified_table(table_name):
    return (
        f"{quote_identifier(settings.db_schema)}."
        f"{quote_identifier(table_name)}"
    )


def sql_literal_list(values):
    return ", ".join("'" + value.replace("'", "''") + "'" for value in values)


database_inspector = inspect(engine)
select_clause_cache = {}


RULE_SOURCE_REQUIREMENTS = {
    SOURCE_TABLE: {
        "id", "fk_employee", "description", "from_date", "to_date",
        "created_at", "status", "transcription_type",
        "column_a", "column_b", "column_c", "ref_do2_no", "do2_type",
        "do2_no", "do2_item_no", "do2_date", "do2_group",
    },
    "employee": {
        "id", "fs_due_date", "fk_status_code", "status_effect_date",
        "created_at", "current_record", "enrolment_date", "birth_date",
    },
    "status_code": {"id", "status_code"},
    "lpc_register": {
        "fk_employee", "discharge_date", "created_at", "army_no", "qe",
        "entry_approved", "fs_approved", "ecs_generated", "record_status",
        "stop_pay_description", "whether_misc_fs", "cancelled",
    },
    "appendix_j": {
        "id", "description", "do2_group", "column_a", "column_b", "column_c",
        "from_date_req", "to_date_req", "rate_req", "tjra_req", "lra_req",
        "ref_do2_no_req",
    },
    "family_detail": {
        "fk_employee", "relationship", "created_at", "record_status",
    },
}

def table_column_names(table_name):
    return {
        column["name"]
        for column in database_inspector.get_columns(
            table_name,
            schema=settings.db_schema,
        )
    }


def validate_rule_sources():
    """Fail early instead of silently creating incorrect rule features."""
    problems = []
    for table_name, required_columns in RULE_SOURCE_REQUIREMENTS.items():
        available_columns = table_column_names(table_name)
        if not available_columns:
            problems.append(f"missing table: {settings.db_schema}.{table_name}")
            continue
        missing_columns = sorted(required_columns - available_columns)
        if missing_columns:
            problems.append(
                f"{settings.db_schema}.{table_name} missing columns: "
                + ", ".join(missing_columns)
            )
    if problems:
        raise RuntimeError("Rule-feature source validation failed:\n- " + "\n- ".join(problems))


validate_rule_sources()


def make_safe_select_clause(table_name, table_alias="d"):
    """Return aliased columns; dates are text to tolerate extreme old dates."""
    cache_key = (table_name, table_alias)
    if cache_key in select_clause_cache:
        return select_clause_cache[cache_key]

    table_columns = database_inspector.get_columns(
        table_name,
        schema=settings.db_schema,
    )
    if not table_columns:
        raise ValueError(f"Table was not found: {settings.db_schema}.{table_name}")

    expressions = []
    converted_date_columns = []
    alias_sql = quote_identifier(table_alias)
    for column in table_columns:
        column_name = column["name"]
        if table_name == SOURCE_TABLE and column_name not in MODEL_SOURCE_COLUMNS:
            continue
        column_sql = f"{alias_sql}.{quote_identifier(column_name)}"
        output_sql = quote_identifier(column_name)
        if isinstance(column["type"], (Date, DateTime)):
            expressions.append(f"{column_sql}::text AS {output_sql}")
            converted_date_columns.append(column_name)
        else:
            expressions.append(f"{column_sql} AS {output_sql}")

    value = (",\n".join(expressions), converted_date_columns)
    select_clause_cache[cache_key] = value
    return value


def build_enriched_query(table_name, where_sql="", limit_sql=""):
    """
    Build one row per selected DO2 with entry-time-safe N/I/MR/V facts.

    All self-history and one-to-many sources are aggregated before they reach
    pandas. Prior DO2 status and processed_date are never read: do2.status is
    mutable and there is no status-history table. Lower id plus an earlier-or-
    equal created_at is a conservative insertion-order proxy because Java can
    reset created_at during pending reprocessing.
    """
    select_clause, date_columns = make_safe_select_clause(table_name, "d")
    do2_history = qualified_table(SOURCE_TABLE)

    excluded_sql = sql_literal_list(DUPLICATE_CHECK_EXCLUSIONS)
    cea_family_sql = sql_literal_list(DUPLICATE_CEA_FAMILY)
    column_ab_sql = sql_literal_list(DUPLICATE_COLUMN_AB)
    mutex_sql = sql_literal_list(MUTEX_DESCRIPTIONS)

    query = f"""
    WITH selected_do2 AS (
        SELECT source_row.*
        FROM {qualified_table(table_name)} AS source_row
        {where_sql}
        {limit_sql}
    )
    SELECT
        {select_clause},

        CASE
            WHEN employee_row.id IS NOT NULL
             AND employee_row.created_at IS NOT NULL
             AND d.created_at IS NOT NULL
             AND employee_row.created_at <= d.created_at
            THEN 1 ELSE 0
        END AS employee_found,
        CASE WHEN employee_row.created_at <= d.created_at
             THEN employee_row.fs_due_date END::text AS employee_fs_due_date,
        CASE WHEN employee_row.created_at <= d.created_at
                   AND (employee_row.status_effect_date IS NULL
                        OR employee_row.status_effect_date <= d.created_at)
             THEN employee_status.status_code END::text AS employee_status_code,
        CASE WHEN employee_row.created_at <= d.created_at
                   AND employee_row.status_effect_date <= d.created_at
             THEN employee_row.status_effect_date END::text
             AS employee_status_effect_date,
        CASE WHEN employee_row.created_at <= d.created_at
             THEN employee_row.created_at END::text AS employee_created_at,
        CASE WHEN employee_row.created_at <= d.created_at
             THEN employee_row.current_record END::text AS employee_current_record,
        CASE WHEN employee_row.created_at <= d.created_at
             THEN employee_row.enrolment_date END::text AS employee_enrolment_date,
        CASE WHEN employee_row.created_at <= d.created_at
             THEN employee_row.birth_date END::text AS employee_birth_date,

        CASE WHEN d.created_at IS NULL THEN 0 ELSE 1 END
            AS rule_history_cutoff_available,
        COALESCE(lpc_facts.prior_lpc_count, 0) AS rule_prior_lpc_count,
        lpc_facts.latest_prior_discharge_date::text
            AS rule_latest_prior_lpc_discharge_date,
        COALESCE(lpc_facts.finalized_lpc_count, 0)
            AS rule_finalized_lpc_count,
        lpc_facts.finalized_lpc_discharge_min::text
            AS rule_finalized_lpc_discharge_min,
        lpc_facts.finalized_lpc_discharge_max::text
            AS rule_finalized_lpc_discharge_max,
        COALESCE(account_closed_facts.account_closed_lpc_count, 0)
            AS rule_account_closed_lpc_count,
        CASE
            WHEN d.army_no IS NOT NULL AND d.qe IS NOT NULL
            THEN 1 ELSE 0
        END AS rule_account_closed_check_available,

        appendix_j_rule.id AS rule_appendix_j_id,
        appendix_j_rule.column_a::text AS rule_appendix_column_a,
        appendix_j_rule.column_b::text AS rule_appendix_column_b,
        appendix_j_rule.column_c::text AS rule_appendix_column_c,
        appendix_j_rule.from_date_req::text AS rule_appendix_from_date_req,
        appendix_j_rule.to_date_req::text AS rule_appendix_to_date_req,
        appendix_j_rule.rate_req::text AS rule_appendix_rate_req,
        appendix_j_rule.tjra_req::text AS rule_appendix_tjra_req,
        appendix_j_rule.lra_req::text AS rule_appendix_lra_req,
        appendix_j_rule.ref_do2_no_req::text AS rule_appendix_ref_do2_no_req,

        CASE
            WHEN EXISTS (
                SELECT 1
                FROM {qualified_table("appendix_j")} AS appendix_exact
                WHERE UPPER(BTRIM(COALESCE(appendix_exact.description::text, '')))
                      = UPPER(BTRIM(COALESCE(d.description::text, '')))
                  AND (
                      appendix_exact.do2_group IS NULL
                      OR BTRIM(appendix_exact.do2_group::text) = ''
                      OR UPPER(BTRIM(appendix_exact.do2_group::text))
                         = UPPER(BTRIM(COALESCE(d.do2_group::text, '')))
                  )
                  AND UPPER(BTRIM(COALESCE(appendix_exact.column_a::text, '')))
                      = UPPER(BTRIM(COALESCE(d.column_a::text, '')))
                  AND UPPER(BTRIM(COALESCE(appendix_exact.column_b::text, '')))
                      = UPPER(BTRIM(COALESCE(d.column_b::text, '')))
                  AND UPPER(BTRIM(COALESCE(appendix_exact.column_c::text, '')))
                      = UPPER(BTRIM(COALESCE(d.column_c::text, '')))
            ) THEN 1 ELSE 0
        END AS rule_appendix_exact_value_match,

        COALESCE(duplicate_facts.prior_duplicate_candidate_count, 0)
            AS rule_prior_duplicate_candidate_count,
        COALESCE(overlap_facts.prior_mutex_overlap_candidate_count, 0)
            AS rule_prior_mutex_overlap_candidate_count,
        COALESCE(reference_facts.reference_match_count, 0)
            AS rule_reference_match_count,

        COALESCE(pay_history.prior_promotion_candidate_count, 0)
            AS rule_prior_promotion_candidate_count,
        COALESCE(pay_history.prior_optcpc_conflict_candidate_count, 0)
            AS rule_prior_optcpc_conflict_candidate_count,
        COALESCE(pay_history.prior_pay_fix_event_candidate_count, 0)
            AS rule_prior_pay_fix_event_candidate_count,

        COALESCE(family_facts.prior_family_member_count, 0)
            AS rule_prior_family_member_count,
        COALESCE(family_facts.prior_missing_relationship_count, 0)
            AS rule_prior_missing_relationship_count

    FROM selected_do2 AS d
    LEFT JOIN {qualified_table("employee")} AS employee_row
        ON employee_row.id = d.fk_employee
    LEFT JOIN {qualified_table("status_code")} AS employee_status
        ON employee_status.id = employee_row.fk_status_code

    LEFT JOIN LATERAL (
        SELECT
            COUNT(*)::bigint AS prior_lpc_count,
            MAX(lpc.discharge_date) AS latest_prior_discharge_date,
            COUNT(*) FILTER (
                WHERE lpc.record_status = 'F'
                  AND lpc.fs_approved IS TRUE
                  AND lpc.ecs_generated IS TRUE
            )::bigint AS finalized_lpc_count,
            MIN(lpc.discharge_date) FILTER (
                WHERE lpc.record_status = 'F'
                  AND lpc.fs_approved IS TRUE
                  AND lpc.ecs_generated IS TRUE
            ) AS finalized_lpc_discharge_min,
            MAX(lpc.discharge_date) FILTER (
                WHERE lpc.record_status = 'F'
                  AND lpc.fs_approved IS TRUE
                  AND lpc.ecs_generated IS TRUE
            ) AS finalized_lpc_discharge_max
        FROM {qualified_table("lpc_register")} AS lpc
        WHERE lpc.fk_employee = d.fk_employee
          AND d.created_at IS NOT NULL
          AND lpc.created_at IS NOT NULL
          AND lpc.created_at < d.created_at
    ) AS lpc_facts ON TRUE

    LEFT JOIN LATERAL (
        SELECT COUNT(*)::bigint AS account_closed_lpc_count
        FROM {qualified_table("lpc_register")} AS lpc_closed
        WHERE lpc_closed.army_no = d.army_no
          AND lpc_closed.qe = d.qe
          AND lpc_closed.entry_approved IS TRUE
          AND lpc_closed.fs_approved IS TRUE
          AND d.created_at IS NOT NULL
          AND lpc_closed.created_at IS NOT NULL
          AND lpc_closed.created_at < d.created_at
    ) AS account_closed_facts ON TRUE

    LEFT JOIN LATERAL (
        SELECT appendix.*
        FROM {qualified_table("appendix_j")} AS appendix
        WHERE UPPER(BTRIM(COALESCE(appendix.description::text, '')))
              = UPPER(BTRIM(COALESCE(d.description::text, '')))
          AND (
              appendix.do2_group IS NULL
              OR BTRIM(appendix.do2_group::text) = ''
              OR UPPER(BTRIM(appendix.do2_group::text))
                 = UPPER(BTRIM(COALESCE(d.do2_group::text, '')))
          )
        ORDER BY
            CASE
                WHEN UPPER(BTRIM(COALESCE(appendix.do2_group::text, '')))
                     = UPPER(BTRIM(COALESCE(d.do2_group::text, '')))
                THEN 0 ELSE 1
            END,
            appendix.id DESC
        LIMIT 1
    ) AS appendix_j_rule ON TRUE

    LEFT JOIN LATERAL (
        SELECT COUNT(history_row.id)::bigint AS prior_duplicate_candidate_count
        FROM {do2_history} AS history_row
        WHERE history_row.fk_employee = d.fk_employee
          AND history_row.id < d.id
          AND d.created_at IS NOT NULL
          AND history_row.created_at IS NOT NULL
          AND history_row.created_at <= d.created_at
          AND UPPER(BTRIM(COALESCE(d.description::text, '')))
              NOT IN ({excluded_sql})
          AND (
              (
                  UPPER(BTRIM(COALESCE(d.description::text, ''))) = 'BUNCH'
                  AND UPPER(BTRIM(COALESCE(history_row.description::text, ''))) = 'BUNCH'
              )
              OR
              (
                  UPPER(BTRIM(COALESCE(d.description::text, ''))) IN ({cea_family_sql})
                  AND history_row.from_date = d.from_date
                  AND UPPER(BTRIM(COALESCE(history_row.description::text, '')))
                      = UPPER(BTRIM(COALESCE(d.description::text, '')))
                  AND history_row.column_a = d.column_a
                  AND history_row.column_b = d.column_b
                  AND history_row.column_c = d.column_c
              )
              OR
              (
                  UPPER(BTRIM(COALESCE(d.description::text, ''))) = 'CANCEL'
                  AND history_row.from_date = d.from_date
                  AND UPPER(BTRIM(COALESCE(history_row.description::text, ''))) = 'CANCEL'
                  AND history_row.column_a = d.column_a
                  AND history_row.ref_do2_no = d.ref_do2_no
              )
              OR
              (
                  UPPER(BTRIM(COALESCE(d.description::text, ''))) IN ({column_ab_sql})
                  AND history_row.from_date = d.from_date
                  AND UPPER(BTRIM(COALESCE(history_row.description::text, '')))
                      = UPPER(BTRIM(COALESCE(d.description::text, '')))
                  AND history_row.column_a = d.column_a
                  AND history_row.column_b = d.column_b
              )
              OR
              (
                  UPPER(BTRIM(COALESCE(d.description::text, '')))
                      NOT IN ('BUNCH', 'CANCEL', {cea_family_sql}, {column_ab_sql})
                  AND history_row.from_date = d.from_date
                  AND UPPER(BTRIM(COALESCE(history_row.description::text, '')))
                      = UPPER(BTRIM(COALESCE(d.description::text, '')))
                  AND history_row.column_a = d.column_a
              )
          )
    ) AS duplicate_facts ON TRUE

    LEFT JOIN LATERAL (
        SELECT COUNT(history_row.id)::bigint AS prior_mutex_overlap_candidate_count
        FROM {do2_history} AS history_row
        WHERE d.from_date IS NOT NULL
          AND UPPER(BTRIM(COALESCE(d.description::text, ''))) IN ({mutex_sql})
          AND history_row.fk_employee = d.fk_employee
          AND history_row.id < d.id
          AND d.created_at IS NOT NULL
          AND history_row.created_at IS NOT NULL
          AND history_row.created_at <= d.created_at
          AND UPPER(BTRIM(COALESCE(history_row.transcription_type::text, ''))) <> 'S'
          AND UPPER(BTRIM(COALESCE(history_row.description::text, ''))) IN ({mutex_sql})
          AND history_row.from_date <= COALESCE(d.to_date, d.from_date)
          AND COALESCE(history_row.to_date, history_row.from_date) >= d.from_date
    ) AS overlap_facts ON TRUE

    LEFT JOIN LATERAL (
        SELECT COUNT(history_row.id)::bigint AS reference_match_count
        FROM {do2_history} AS history_row
        WHERE d.ref_do2_no IS NOT NULL
          AND d.created_at IS NOT NULL
          AND history_row.created_at IS NOT NULL
          AND history_row.created_at <= d.created_at
          AND history_row.id < d.id
          AND history_row.fk_employee = d.fk_employee
          AND BTRIM(d.ref_do2_no::text) = CONCAT(
              BTRIM(history_row.do2_type::text), '/',
              LPAD(BTRIM(history_row.do2_no::text), 4, '0'), '/',
              LPAD(BTRIM(history_row.do2_item_no::text), 3, '0'), '/',
              EXTRACT(YEAR FROM history_row.do2_date)::integer::text
          )
    ) AS reference_facts ON TRUE

    LEFT JOIN LATERAL (
        SELECT
            COUNT(history_row.id) FILTER (
                WHERE UPPER(BTRIM(COALESCE(history_row.description::text, '')))
                      IN ('PROMOT', 'MACPS')
                  AND history_row.from_date = d.from_date
            )::bigint AS prior_promotion_candidate_count,
            COUNT(history_row.id) FILTER (
                WHERE UPPER(BTRIM(COALESCE(history_row.description::text, ''))) = 'OPTCPC'
                  AND UPPER(BTRIM(COALESCE(history_row.column_b::text, ''))) = 'SEVENPC'
                  AND UPPER(BTRIM(COALESCE(history_row.column_a::text, ''))) <> 'INCR'
                  AND history_row.from_date IN (d.from_date, d.to_date)
            )::bigint AS prior_optcpc_conflict_candidate_count,
            COUNT(history_row.id) FILTER (
                WHERE UPPER(BTRIM(COALESCE(history_row.description::text, '')))
                      IN ('OPTFIX', 'PROFIX', 'ACPFIX', 'HAVFIX')
            )::bigint AS prior_pay_fix_event_candidate_count
        FROM {do2_history} AS history_row
        WHERE UPPER(BTRIM(COALESCE(d.description::text, '')))
                  IN ('OPTFIX', 'PROFIX', 'POSIN', 'OPTCPC')
          AND history_row.fk_employee = d.fk_employee
          AND history_row.id < d.id
          AND d.created_at IS NOT NULL
          AND history_row.created_at IS NOT NULL
          AND history_row.created_at <= d.created_at
    ) AS pay_history ON TRUE

    LEFT JOIN LATERAL (
        SELECT
            COUNT(*)::bigint AS prior_family_member_count,
            COUNT(*) FILTER (
                WHERE family_row.relationship IS NULL
                   OR BTRIM(family_row.relationship::text) = ''
            )::bigint AS prior_missing_relationship_count
        FROM {qualified_table("family_detail")} AS family_row
        WHERE family_row.fk_employee = d.fk_employee
          AND d.created_at IS NOT NULL
          AND family_row.created_at IS NOT NULL
          AND family_row.created_at < d.created_at
          AND UPPER(BTRIM(COALESCE(family_row.record_status::text, ''))) NOT IN ('D', 'DELETED')
    ) AS family_facts ON TRUE
    """
    return text(query), date_columns


def sanitize_timestamp_text(series: pd.Series) -> pd.Series:
    """Null out PostgreSQL 'infinity'/'-infinity' sentinel timestamps.

    Columns are cast to text in SQL to tolerate extreme dates, but
    Postgres can also store the literal 'infinity' / '-infinity'
    values, which pandas cannot parse and turns into NaT with no
    explanation. Handling this explicitly here (instead of only in
    the validation step) lets us report and act on it deliberately.
    """
    cleaned = series.astype("string").str.strip().str.lower()
    sentinel_mask = cleaned.isin({"infinity", "-infinity"})
    result = series.copy()
    result.loc[sentinel_mask.fillna(False)] = None
    return result


def parse_datetime_mixed(series: pd.Series) -> pd.Series:
    """Parse mixed PostgreSQL date/timestamp text without format inference loss."""
    sanitized = sanitize_timestamp_text(series)
    try:
        return pd.to_datetime(sanitized, errors="coerce", format="mixed")
    except TypeError:
        # Compatibility fallback for older pandas releases without format="mixed".
        return pd.to_datetime(sanitized, errors="coerce")


def timestamp_quality_masks(series: pd.Series, parsed: pd.Series) -> dict[str, pd.Series]:
    """Separate missing, sentinel, and malformed timestamps for reporting."""
    text_value = series.astype("string").str.strip()
    missing = series.isna() | text_value.eq("").fillna(False)
    sentinel = text_value.str.lower().isin({"infinity", "-infinity"}).fillna(False)
    rejected = parsed.isna()
    malformed = rejected & ~missing & ~sentinel
    return {
        "missing": missing,
        "sentinel": sentinel,
        "malformed": malformed,
        "rejected": rejected,
    }


def normalized_status_sql(alias):
    return (
        "UPPER(BTRIM(REPLACE(COALESCE("
        + alias
        + ".status::text, ''), '\"', '')))"
    )


def load_training_population_prior():
    status_expression = normalized_status_sql("source_row")
    query = text(f"""
        SELECT {status_expression} AS status, COUNT(*)::bigint AS rows
        FROM {qualified_table(SOURCE_TABLE)} AS source_row
        WHERE source_row.{quote_identifier(SPLIT_DATE_COLUMN)} >= CAST(:start_date AS timestamp)
          AND source_row.{quote_identifier(SPLIT_DATE_COLUMN)} < CAST(:end_date AS timestamp)
          AND {status_expression} IN ({MODEL_STATUS_SQL})
          AND UPPER(BTRIM(COALESCE(source_row.do2_group::text, '')))
              IN ({sql_literal_list(VALID_DO2_GROUPS)})
        GROUP BY 1
        ORDER BY 1
    """)
    return pd.read_sql_query(
        query,
        engine,
        params={"start_date": TRAIN_START, "end_date": TRAIN_END},
    )


def load_training_data():
    status_expression = normalized_status_sql("source_row")
    v_status_expression = normalized_status_sql("v_source")
    valid_groups_sql = sql_literal_list(VALID_DO2_GROUPS)

    # All non-V rows are retained. V is sampled only inside the training window.
    where_sql = f"""
        WHERE source_row.{quote_identifier(SPLIT_DATE_COLUMN)} >= CAST(:start_date AS timestamp)
          AND source_row.{quote_identifier(SPLIT_DATE_COLUMN)} < CAST(:end_date AS timestamp)
          AND {status_expression} IN ({LOAD_STATUS_SQL})
          AND UPPER(BTRIM(COALESCE(source_row.do2_group::text, '')))
              IN ({valid_groups_sql})
          AND (
              {status_expression} <> 'V'
              OR source_row.id IN (
                  SELECT sampled_v.id
                  FROM (
                      SELECT v_source.id
                      FROM {qualified_table(SOURCE_TABLE)} AS v_source
                      WHERE v_source.{quote_identifier(SPLIT_DATE_COLUMN)} >= CAST(:start_date AS timestamp)
                        AND v_source.{quote_identifier(SPLIT_DATE_COLUMN)} < CAST(:end_date AS timestamp)
                        AND {v_status_expression} = 'V'
                        AND UPPER(BTRIM(COALESCE(v_source.do2_group::text, '')))
                            IN ({valid_groups_sql})
                      ORDER BY md5(COALESCE(v_source.id::text, ''))
                      LIMIT :max_train_v_rows
                  ) AS sampled_v
              )
          )
    """
    query, date_columns = build_enriched_query(
        SOURCE_TABLE,
        where_sql=where_sql,
    )
    frame = pd.read_sql_query(
        query,
        engine,
        params={
            "start_date": TRAIN_START,
            "end_date": TRAIN_END,
            "max_train_v_rows": int(MAX_TRAIN_V_ROWS),
        },
    )
    print("Training date columns loaded as text:", date_columns)
    print("Training rows enriched with status-independent prior candidate facts.")
    return frame


def load_holdout_data(start_date, end_date, max_rows=None):
    params = {"start_date": start_date, "end_date": end_date}
    status_expression = normalized_status_sql("source_row")
    where_sql = f"""
        WHERE source_row.{quote_identifier(SPLIT_DATE_COLUMN)} >= CAST(:start_date AS timestamp)
          AND source_row.{quote_identifier(SPLIT_DATE_COLUMN)} < CAST(:end_date AS timestamp)
          AND {status_expression} IN ({LOAD_STATUS_SQL})
          AND UPPER(BTRIM(COALESCE(source_row.do2_group::text, '')))
              IN ({sql_literal_list(VALID_DO2_GROUPS)})
    """
    limit_sql = ""
    if max_rows is not None:
        limit_sql = """
            ORDER BY md5(COALESCE(source_row.id::text, ''))
            LIMIT :max_rows
        """
        params["max_rows"] = int(max_rows)

    query, date_columns = build_enriched_query(
        SOURCE_TABLE,
        where_sql=where_sql,
        limit_sql=limit_sql,
    )
    frame = pd.read_sql_query(query, engine, params=params)
    print("Holdout date columns loaded as text:", date_columns)
    print("Holdout rows enriched with status-independent prior candidate facts.")
    return frame


def enriched_cache_path(split_name, start_date, end_date, row_cap):
    cap_text = "all" if row_cap is None else str(int(row_cap))
    filename = (
        f"{DATA_CACHE_VERSION}_{split_name}_"
        f"{start_date}_{end_date}_{cap_text}.parquet"
    ).replace(":", "-")
    return DATA_CACHE_DIR / filename


def load_or_cache_enriched(
    split_name,
    start_date,
    end_date,
    row_cap,
    loader,
):
    cache_path = enriched_cache_path(
        split_name, start_date, end_date, row_cap
    )
    if USE_DATA_CACHE and cache_path.exists() and not REFRESH_DATA_CACHE:
        print(f"Loading cached {split_name} data:", cache_path)
        started = time.perf_counter()
        frame = pd.read_parquet(cache_path)
        print(
            f"Loaded cached {split_name} rows in "
            f"{time.perf_counter() - started:,.1f} seconds"
        )
        return frame

    started = time.perf_counter()
    frame = loader()
    print(
        f"Database enrichment for {split_name} finished in "
        f"{(time.perf_counter() - started) / 60:,.1f} minutes"
    )
    if USE_DATA_CACHE:
        DATA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        frame.to_parquet(cache_path, index=False)
        print(f"Cached enriched {split_name} data:", cache_path)
    return frame


def show_recommended_index_sql():
    print("Recommended one-time PostgreSQL indexes for faster enrichment:")
    print(f"""
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_do2_ml_employee_created_id
    ON {qualified_table(SOURCE_TABLE)} (fk_employee, created_at, id);
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_do2_ml_split_selection
    ON {qualified_table(SOURCE_TABLE)} (created_at, do2_group, id);
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_do2_ml_employee_desc_from_created
    ON {qualified_table(SOURCE_TABLE)} (fk_employee, description, from_date, created_at);
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_lpc_ml_employee_created
    ON {qualified_table('lpc_register')} (fk_employee, created_at);
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_family_ml_employee_created
    ON {qualified_table('family_detail')} (fk_employee, created_at);
""".strip())


training_population_prior = load_training_population_prior()
show_recommended_index_sql()
train_raw = load_or_cache_enriched(
    "train",
    TRAIN_START,
    TRAIN_END,
    MAX_TRAIN_V_ROWS,
    load_training_data,
)
validation_raw = load_or_cache_enriched(
    "validation",
    VALIDATION_START,
    VALIDATION_END,
    MAX_VALIDATION_ROWS,
    lambda: load_holdout_data(
        start_date=VALIDATION_START,
        end_date=VALIDATION_END,
        max_rows=MAX_VALIDATION_ROWS,
    ),
)

print("Raw training shape:", train_raw.shape)
print("Raw validation shape:", validation_raw.shape)
display(training_population_prior)


Recommended one-time PostgreSQL indexes for faster enrichment:
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_do2_ml_employee_created_id
    ON "public"."do2" (fk_employee, created_at, id);
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_do2_ml_split_selection
    ON "public"."do2" (created_at, do2_group, id);
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_do2_ml_employee_desc_from_created
    ON "public"."do2" (fk_employee, description, from_date, created_at);
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_lpc_ml_employee_created
    ON "public"."lpc_register" (fk_employee, created_at);
CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_family_ml_employee_created
    ON "public"."family_detail" (fk_employee, created_at);
Loading cached train data: /home/ruthikreddy/Desktop/do2/do2_status_prediction/data/model_cache/time_safe_v7_java_n_features_no_pattern_model_train_2022-01-01_2025-01-01_250000.parquet
Loaded cached train rows in 0.9 seconds
Loading cached validation data: /home/ruthikreddy/Desktop/do2/d

,status,rows
0,I,82802
1,MR,32380
2,N,7084
3,V,2110131


## 4. Clean labels without target-dependent holdout filtering

In [6]:
def normalize_status(series: pd.Series) -> pd.Series:
    """Standardize status values before filtering."""
    return (
        series.fillna("")
        .astype("string")
        .str.replace('"', "", regex=False)
        .str.strip()
        .str.upper()
    )


REQUIRED_V_COLUMNS = (
    "unit_code", "army_no", "fk_employee", "do2_group", "do2_type",
    "do2_no", "do2_item_no", "do2_date", "created_at", "description",
    "transcription_type", "qe", "check_digit",
)


def missing_value_mask(series):
    missing = series.isna()
    if series.dtype == "object" or str(series.dtype).startswith("string"):
        missing = missing | series.fillna("").astype(str).str.strip().eq("")
    return missing


def clean_split(frame, split_name, *, drop_incomplete_v_labels=False):
    cleaned = frame.copy()
    initial_rows = len(cleaned)
    cleaned["status"] = normalize_status(cleaned["status"])

    # Rows whose created_at cannot be parsed (NULL, malformed, or a
    # Postgres infinity/-infinity sentinel) cannot be placed in a
    # time-safe split. Remove them here, explicitly, with a count in the
    # report — instead of letting them silently become NaT and blow up
    # the split_validate check downstream.
    created_at_original = cleaned[SPLIT_DATE_COLUMN].copy()
    created_at_parsed = parse_datetime_mixed(created_at_original)
    timestamp_masks = timestamp_quality_masks(
        created_at_original,
        created_at_parsed,
    )
    unparseable_created_at_rows = int(timestamp_masks["rejected"].sum())
    missing_created_at_rows = int(timestamp_masks["missing"].sum())
    sentinel_created_at_rows = int(timestamp_masks["sentinel"].sum())
    malformed_created_at_rows = int(timestamp_masks["malformed"].sum())

    if unparseable_created_at_rows:
        rejected_examples = (
            created_at_original.loc[timestamp_masks["rejected"]]
            .astype("string")
            .fillna("<NULL>")
            .value_counts(dropna=False)
            .head(10)
            .rename_axis("rejected_created_at")
            .reset_index(name="rows")
        )
        print(
            f"{split_name}: rejected {unparseable_created_at_rows:,} created_at "
            "values after mixed-format parsing; top examples:"
        )
        display(rejected_examples)

    if DROP_UNPARSEABLE_CREATED_AT_ROWS:
        cleaned = cleaned.loc[created_at_parsed.notna()].copy()
    cleaned[SPLIT_DATE_COLUMN] = created_at_parsed.loc[cleaned.index]

    valid_status_mask = cleaned["status"].isin(VALID_STATUSES)
    excluded_status_rows = int((~valid_status_mask).sum())

    excluded_lifecycle_status_rows = int(

        cleaned["status"].isin(set(LOAD_STATUSES) - set(VALID_STATUSES)).sum()

    )

    cleaned = cleaned.loc[valid_status_mask].copy()


    normalized_group = (
        cleaned["do2_group"].fillna("").astype("string").str.strip().str.upper()
    )
    valid_group_mask = normalized_group.isin(VALID_DO2_GROUPS)
    excluded_do2_group_rows = int((~valid_group_mask).sum())
    cleaned = cleaned.loc[valid_group_mask].copy()

    missing_required = pd.DataFrame(
        {
            column: missing_value_mask(cleaned[column])
            for column in REQUIRED_V_COLUMNS
            if column in cleaned.columns
        },
        index=cleaned.index,
    ).any(axis=1)

    incomplete_v_rows_found = int(
        (cleaned["status"].eq("V") & missing_required).sum()
    )
    incomplete_v_rows_removed = 0
    if drop_incomplete_v_labels:
        remove_mask = cleaned["status"].eq("V") & missing_required
        incomplete_v_rows_removed = int(remove_mask.sum())
        cleaned = cleaned.loc[~remove_mask].copy()

    cleaned = cleaned.reset_index(drop=True)
    report = {
        "split": split_name,
        "initial_rows": initial_rows,
        "unparseable_created_at_rows": unparseable_created_at_rows,
        "missing_created_at_rows": missing_created_at_rows,
        "sentinel_created_at_rows": sentinel_created_at_rows,
        "malformed_created_at_rows": malformed_created_at_rows,
        "excluded_status_rows": excluded_status_rows,
        "excluded_lifecycle_status_rows": excluded_lifecycle_status_rows,
        "excluded_do2_group_rows": excluded_do2_group_rows,
        "incomplete_v_rows_found": incomplete_v_rows_found,
        "incomplete_v_rows_removed": incomplete_v_rows_removed,
        "final_rows": len(cleaned),
    }
    return cleaned, report


train_df, train_cleaning_report = clean_split(
    train_raw,
    "train",
    drop_incomplete_v_labels=DROP_INCOMPLETE_V_TRAINING_LABELS,
)
validation_df, validation_cleaning_report = clean_split(
    validation_raw,
    "validation",
    drop_incomplete_v_labels=False,
)

cleaning_report = pd.DataFrame(
    [train_cleaning_report, validation_cleaning_report]
)
display(cleaning_report)

del train_raw, validation_raw
_ = gc.collect()


,split,initial_rows,unparseable_created_at_rows,missing_created_at_rows,sentinel_created_at_rows,malformed_created_at_rows,excluded_status_rows,excluded_lifecycle_status_rows,excluded_do2_group_rows,incomplete_v_rows_found,incomplete_v_rows_removed,final_rows
0,train,597079,0,0,0,0,224813,224813,0,0,0,372266
1,validation,250000,0,0,0,0,12255,12255,0,1149,0,237745


## 5. Validate split grain, dates, classes, and overlap

In [7]:
def split_profile(frame, split_name):
    return (
        frame["status"]
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="rows")
        .assign(
            split=split_name,
            percentage=lambda value: value["rows"] / value["rows"].sum() * 100,
        )[["split", "status", "rows", "percentage"]]
    )


def validate_split(frame, split_name, start_date, end_date):
    if "id" not in frame:
        raise ValueError(f"{split_name} is missing id")
    duplicate_ids = int(frame["id"].duplicated().sum())
    if duplicate_ids:
        raise ValueError(
            f"{split_name} contains {duplicate_ids:,} duplicated ids after joins"
        )

    timestamps = pd.to_datetime(frame[SPLIT_DATE_COLUMN], errors="coerce")
    missing_timestamps = int(timestamps.isna().sum())
    outside_window = int(
        (~timestamps.between(
            pd.Timestamp(start_date),
            pd.Timestamp(end_date),
            inclusive="left",
        )).sum()
    )
    if missing_timestamps or outside_window:
        raise ValueError(
            f"{split_name} timestamp validation failed: "
            f"missing={missing_timestamps:,}, outside_window={outside_window:,}"
        )
    return {
        "split": split_name,
        "rows": int(len(frame)),
        "unique_ids": int(frame["id"].nunique()),
        "minimum_created_at": timestamps.min(),
        "maximum_created_at": timestamps.max(),
    }


split_checks = pd.DataFrame([
    validate_split(train_df, "train", TRAIN_START, TRAIN_END),
    validate_split(
        validation_df, "validation", VALIDATION_START, VALIDATION_END
    ),
])

TRAIN_IDS = frozenset(train_df["id"].tolist())
VALIDATION_IDS = frozenset(validation_df["id"].tolist())
train_validation_overlap = len(TRAIN_IDS & VALIDATION_IDS)
if train_validation_overlap:
    raise ValueError(
        f"Train/validation id overlap: {train_validation_overlap:,} rows"
    )

status_profile = pd.concat(
    [
        split_profile(train_df, "train"),
        split_profile(validation_df, "validation"),
    ],
    ignore_index=True,
)
display(split_checks)
display(status_profile)
print("Train/validation id overlap:", train_validation_overlap)


,split,rows,unique_ids,minimum_created_at,maximum_created_at
0,train,372266,372266,2022-01-03 17:04:15.160000,2024-12-31 16:25:23.533
1,validation,237745,237745,2025-01-01 10:44:56.681978,2025-06-30 18:24:29.335


,split,status,rows,percentage
0,train,V,250000,67.156281
1,train,I,82802,22.242697
2,train,MR,32380,8.698081
3,train,N,7084,1.90294
4,validation,V,187687,78.944668
5,validation,I,48305,20.317988
6,validation,MR,1122,0.471934
7,validation,N,631,0.26541


Train/validation id overlap: 0


## 6. Build one reproducible feature set

In [ ]:

from dataclasses import dataclass

import numpy as np
import pandas as pd


FEATURE_SET_VERSION = "stable_bill_pattern_features_v9_verified_java_rules"
MISSING_CATEGORY = "__MISSING__"


LEAKAGE_COLUMNS = frozenset(
    {
        "status", "target", "is_non_valid", "reason", "sys_reason",
        "change_reason", "rejection_type", "approved", "reviewed",
        "accepted_qe", "review_qe", "fk_auditor", "fk_aao", "fk_ao",
        "fk_hoo", "auditor_date", "aao_date", "ao_date", "hoo_date",
        "processed_date", "bulk_approval", "attachment_verified",
        "udise_aud_verified", "udise_aao_verified", "udise_sao_verified",
        "fk_remove_mr_ao", "fk_remove_mr_ao_date", "mr_removal_reason",
        "fk_put_mr_ao", "fk_put_mr_ao_date", "mr_put_reason", "fk_task",
        "sms_sent_at", "pan", "sms_sent", "fk_usr", "remarks", "audit_cage",
    }
)


# Only columns needed for cleaning, verified rules, or point-in-time joins.
CANDIDATE_COLUMNS = [
    "unit_code", "fk_unit", "army_no", "fk_employee", "check_digit",
    "do2_group", "do2_type", "do2_no", "do2_item_no", "do2_date",
    "description", "from_date", "to_date", "rate", "tjra", "lra",
    "column_a", "column_b", "column_c", "ref_do2_no",
    "transcription_type", "qe", "created_at", "attachment", "delayed",
    "time_bar_sanction", "mr",
]


# Retained for V-label cleaning and source joins even when not direct features.
REQUIRED_INPUT_COLUMNS = [
    "unit_code", "army_no", "fk_employee", "do2_group", "do2_type",
    "do2_no", "do2_item_no", "do2_date", "created_at", "description",
    "transcription_type", "qe", "check_digit",
]


# Stable business categories only; identifiers and QE are not direct features.
CATEGORICAL_BASE = [
    "do2_group", "do2_type", "description", "transcription_type",
    "delayed", "time_bar_sanction",
]
NUMERIC_BASE = ["rate", "tjra", "lra"]
TEXT_BASE: list[str] = []

FEATURE_CONFIG = {
    "candidate_columns": CANDIDATE_COLUMNS,
    "required_input_columns": REQUIRED_INPUT_COLUMNS,
    "categorical_base": CATEGORICAL_BASE,
    "numeric_base": NUMERIC_BASE,
    "text_base": TEXT_BASE,
}

# Backward-compatible audit list used by the notebook metadata/report cells.
# These names are no longer generated, rather than generated and then dropped.
FEATURES_TO_DROP = frozenset(
    {
        "do2_item_no",
        "unit_code",
        "check_digit",
        "punishment",
        "qe",
        "do2_group_type",
        "unit_do2_type",
        "description_do2_type",
        "description_char_count",
        "description_word_count",
        "description_has_digit",
        "rate_is_negative",
        "tjra_is_negative",
        "lra_is_negative",
        "rate_log_abs",
        "tjra_log_abs",
        "lra_log_abs",
        "rate_tjra_abs_diff",
        "rate_lra_abs_diff",
        "do2_date_plausible",
        "from_date_plausible",
        "to_date_plausible",
        "created_at_plausible",
        "army_no_is_placeholder",
        "abs_duration_days",
        "log_abs_duration_days",
        "abs_submission_delay_days",
        "log_abs_submission_delay_days",
        "created_before_do2",
        "same_day_created_do2",
        "zero_duration",
        "long_duration",
        "core_required_present_count",
        "conditional_required_present_count",
        "conditional_required_missing_count",
        "has_punishment",
        "has_pto_remarks",
        "has_column9_remarks",
        "has_cfa_sanction_received",
        "has_cfa_sanction_letter_no",
        "has_cfa_sanction_date",
        "prior_batch_row_count",
        "prior_do2_no_row_count",
        "prior_army_do2_type_row_count",
        "prior_unit_do2_type_row_count",
        "prior_batch_do2_item_share",
    }
)

CORE_REQUIRED_COLUMNS = (
    "army_no", "check_digit", "fk_employee", "unit_code", "fk_unit",
    "do2_group", "do2_type", "do2_no", "do2_item_no", "do2_date",
    "description", "from_date", "transcription_type",
)

APPLICATION_EFFECTIVE_STATUS_CODES = frozenset({"E", "N", "R", "D", "M", "W", "T"})
ABOLISHED_DESCRIPTION_SET = frozenset(
    {
        "CILQ", "SNLQ", "FAA", "STRINC", "HCA", "FLCHCA", "ADALLC",
        "ACTGA", "COMNDO", "INSTAL", "OUTFIT", "HSINC", "HCWA",
        "CONDMT", "BOUNTY", "RLCA1", "RLCA2", "RLCA3", "RLCA4",
        "TYOJ", "TYRJ", "DAILY",
    }
)


# Documentation only: deliberately excluded weak/redundant groups.
REMOVED_FEATURE_GROUPS = (
    "identifier-derived and in-memory history features",
    "description character/word/digit counts",
    "raw categorical pair interactions",
    "log/absolute amount transforms and amount-difference proxies",
    "absolute/log duration transforms",
    "QE/input-QE parsing and quarter-distance features",
    "global conditional-field missing counts",
    "free-text remark presence flags",
    "unused CFA sanction and punishment fields",
    "duplicate rule counts when only existence is needed",
    "unscoped service-age thresholds",
)


@dataclass
class FeatureResult:
    frame: pd.DataFrame
    categorical_columns: list[str]
    text_columns: list[str]


def normalize_status(series: pd.Series) -> pd.Series:
    return (
        series.fillna("")
        .astype("string")
        .str.replace('"', "", regex=False)
        .str.strip()
        .str.upper()
    )


def supervised_status_mask(frame: pd.DataFrame) -> pd.Series:
    if "status" not in frame:
        raise KeyError("status is required for supervised training/evaluation")
    return normalize_status(frame["status"]).ne("")


def filter_supervised_rows(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.loc[supervised_status_mask(frame)].reset_index(drop=True)


def leakage_columns_present(columns: list[str] | pd.Index) -> list[str]:
    present = set(columns)
    return sorted(column for column in LEAKAGE_COLUMNS if column in present)


def validate_prediction_columns(frame: pd.DataFrame, required_columns: list[str]) -> None:
    missing = sorted(set(required_columns) - set(frame.columns))
    if missing:
        raise ValueError(
            "Prediction input is missing required entry-time columns: "
            + ", ".join(missing)
        )


def _series(frame: pd.DataFrame, column: str, default=np.nan) -> pd.Series:
    if column in frame:
        return frame[column]
    return pd.Series(default, index=frame.index)


def _clean_category(series: pd.Series) -> pd.Series:
    cleaned = series.astype("string").str.strip().str.upper()
    return cleaned.fillna(MISSING_CATEGORY).replace("", MISSING_CATEGORY).astype(str)


def _normalized_text(frame: pd.DataFrame, column: str) -> pd.Series:
    return (
        _series(frame, column, "")
        .fillna("")
        .astype("string")
        .str.strip()
        .str.upper()
    )


def _present(series: pd.Series) -> pd.Series:
    return (
        series.notna()
        & series.astype("string").str.strip().fillna("").ne("")
    ).astype("int8")


def _column_present(frame: pd.DataFrame, column: str) -> pd.Series:
    return _present(_series(frame, column))


def _numeric_source(frame: pd.DataFrame, column: str) -> pd.Series:
    return pd.to_numeric(_series(frame, column), errors="coerce").astype("float32")


def _truthy_source(frame: pd.DataFrame, column: str) -> pd.Series:
    return _normalized_text(frame, column).isin(
        {"1", "TRUE", "T", "YES", "Y", "REQUIRED", "R"}
    )


def _date(frame: pd.DataFrame, column: str) -> pd.Series:
    return parse_datetime_mixed(_series(frame, column))


def _rule_bool(condition: pd.Series, available: pd.Series | None = None) -> pd.Series:
    output = condition.astype("float32")
    if available is not None:
        output = output.where(available, np.nan)
    return output.astype("float32")


def build_status_n_features(frame):
    features = pd.DataFrame(index=frame.index)

    from_date = _date(frame, "from_date")
    to_date = _date(frame, "to_date")
    created_at = _date(frame, "created_at")
    employee_fs_due_date = _date(frame, "employee_fs_due_date")
    lpc_discharge_date = _date(frame, "rule_latest_prior_lpc_discharge_date")
    status_effect_date = _date(frame, "employee_status_effect_date")
    employee_created_at = _date(frame, "employee_created_at")

    employee_found = _numeric_source(frame, "employee_found").fillna(0).gt(0)
    status_code = (
        frame.get("employee_status_code", pd.Series("", index=frame.index))
        .fillna("")
        .astype("string")
        .str.strip()
        .str.upper()
    )
    status_available = employee_found & status_code.ne("")

    # The processor uses an LPC discharge date as an FS-date fallback.
    effective_fs_due_date = employee_fs_due_date.fillna(lpc_discharge_date)
    fs_date_available = from_date.notna() & effective_fs_due_date.notna()
    to_fs_date_available = to_date.notna() & effective_fs_due_date.notna()

    features["n_employee_found"] = employee_found.astype("int8")
    features["n_employee_status_missing"] = (~status_available).astype("int8")
    features["n_employee_effective_by_application_rule"] = _rule_bool(
        status_code.isin(APPLICATION_EFFECTIVE_STATUS_CODES),
        status_available,
    )
    features["n_employee_not_effective_by_application_rule"] = _rule_bool(
        ~status_code.isin(APPLICATION_EFFECTIVE_STATUS_CODES),
        status_available,
    )

    features["n_fs_due_date_missing"] = employee_fs_due_date.isna().astype("int8")
    features["n_fs_due_date_from_prior_lpc"] = (
        employee_fs_due_date.isna() & lpc_discharge_date.notna()
    ).astype("int8")
    features["n_effective_fs_due_date_missing"] = effective_fs_due_date.isna().astype("int8")
    features["n_days_from_fs_due_date"] = (
        from_date.dt.normalize() - effective_fs_due_date.dt.normalize()
    ).dt.days.astype("float32")
    features["n_days_after_fs_due_date"] = (
        features["n_days_from_fs_due_date"].clip(lower=0).astype("float32")
    )
    features["n_from_date_after_fs_due_date"] = _rule_bool(
        from_date.gt(effective_fs_due_date),
        fs_date_available,
    )
    features["n_from_date_within_fs_month"] = _rule_bool(
        from_date.dt.to_period("M").eq(effective_fs_due_date.dt.to_period("M")),
        fs_date_available,
    )
    features["n_to_date_after_fs_due_date"] = _rule_bool(
        to_date.gt(effective_fs_due_date),
        to_fs_date_available,
    )

    description = _clean_category(
        frame.get("description", pd.Series("", index=frame.index))
    )
    features["n_description_is_death"] = description.eq("DEATH").astype("int8")
    features["n_description_is_atcorr"] = description.eq("ATCORR").astype("int8")
    features["n_description_is_optfix"] = description.eq("OPTFIX").astype("int8")
    features["n_description_is_cea"] = description.isin({"CEA", "CEAHC"}).astype("int8")
    features["n_prior_lpc_count"] = _numeric_source(frame, "rule_prior_lpc_count").fillna(0)
    features["n_prior_lpc_exists"] = features["n_prior_lpc_count"].gt(0).astype("int8")
    features["n_finalized_lpc_count"] = _numeric_source(
        frame, "rule_finalized_lpc_count"
    ).fillna(0).astype("float32")
    features["n_finalized_lpc_exists"] = (
        features["n_finalized_lpc_count"].gt(0).astype("int8")
    )
    finalized_min = _date(frame, "rule_finalized_lpc_discharge_min")
    finalized_max = _date(frame, "rule_finalized_lpc_discharge_max")
    features["n_finalized_lpc_span_days"] = (
        finalized_max - finalized_min
    ).dt.days.astype("float32")
    features["n_lpc_first_row_nondeterministic"] = (
        features["n_finalized_lpc_count"].gt(1).astype("int8")
    )
    features["n_account_closed_check_available"] = _numeric_source(
        frame, "rule_account_closed_check_available"
    ).fillna(0).astype("int8")
    features["n_account_closed_lpc_count"] = _numeric_source(
        frame, "rule_account_closed_lpc_count"
    ).fillna(0).astype("float32")
    features["n_account_closed_bill_qe_match"] = (
        features["n_account_closed_lpc_count"].gt(0).astype("int8")
    )

    return features


MUTEX_DESCRIPTION_SET = frozenset({
    "TYOJ", "TYRJ", "DRDSRA", "AL", "CL", "SL", "EXTNAL", "EXTNCL",
    "EXTNSL", "TJRA", "RTYOJ", "EXTNLV", "RTYRJ", "RDAILY", "HATRIP",
    "MRA", "FRA", "SRA", "GSRA", "NRA", "ISNRA", "ISMRA", "ISFRA",
    "ISHRA", "SPLLVE", "PRPLVE", "PAL", "BAL", "ADVAL", "ACCMAL",
    "RL", "DAILY", "POSDA", "LDAILY",
})
DUPLICATE_CEA_FAMILY_SET = frozenset({
    "CEA", "CEAHC", "HOSSUB", "HSBTWN", "CHNOK", "CHSRNM", "CHFMNM",
})
DUPLICATE_COLUMN_AB_SET = frozenset({"SCCIA", "MEDAL", "UID", "BIOMET"})
DUPLICATE_EXCLUDED_SET = frozenset({"ATCORR", "LRA", "DA", "CEATWN", "NOK"})


def build_status_i_features(frame):
    features = pd.DataFrame(index=frame.index)
    description = _clean_category(
        frame.get("description", pd.Series("", index=frame.index))
    )
    from_date = _date(frame, "from_date")
    employee_fs_due_date = _date(frame, "employee_fs_due_date")
    enrolment_date = _date(frame, "employee_enrolment_date")
    birth_date = _date(frame, "employee_birth_date")
    transcription_type = _normalized_text(frame, "transcription_type")
    column_a = _normalized_text(frame, "column_a")
    army_no_present = _column_present(frame, "army_no").gt(0)
    description_present = _column_present(frame, "description").gt(0)

    # Verified save/update and Java processor rules. These use only values
    # available on the incoming DO2 row plus the point-in-time employee DOB.
    features["i_army_no_missing_at_save"] = (~army_no_present).astype("int8")
    features["i_description_missing_at_save"] = (
        ~description_present
    ).astype("int8")
    features["i_daa_always_inadmissible"] = description.eq("DAA").astype("int8")
    features["i_bunch_wrong_transcription_type"] = (
        description.eq("BUNCH") & transcription_type.ne("T")
    ).astype("int8")

    cilq_granted = description.eq("CILQ") & column_a.eq("GRANTED")
    cilq_age_days = (from_date - birth_date).dt.days.astype("float32")
    features["i_cilq_granted_birth_date_missing"] = (
        cilq_granted & birth_date.isna()
    ).astype("int8")
    features["i_cilq_granted_from_date_missing"] = (
        cilq_granted & from_date.isna()
    ).astype("int8")
    features["i_cilq_granted_age_below_25_on_from_date"] = _rule_bool(
        cilq_age_days.lt(25 * 365.2425),
        cilq_granted & birth_date.notna() & from_date.notna(),
    )

    features["i_description_abolished_on_from_date"] = (
        description.isin(ABOLISHED_DESCRIPTION_SET)
        & from_date.ge(pd.Timestamp("2017-07-01"))
    ).astype("int8")
    duplicate_group = pd.Series("DEFAULT", index=frame.index, dtype="object")
    duplicate_group.loc[description.isin(DUPLICATE_EXCLUDED_SET)] = "EXCLUDED"
    duplicate_group.loc[description.eq("BUNCH")] = "BUNCH"
    duplicate_group.loc[description.isin(DUPLICATE_CEA_FAMILY_SET)] = "CEA_FAMILY"
    duplicate_group.loc[description.eq("CANCEL")] = "CANCEL"
    duplicate_group.loc[description.isin(DUPLICATE_COLUMN_AB_SET)] = "COLUMN_AB"
    features["i_duplicate_rule_group"] = duplicate_group
    features["i_duplicate_check_applicable"] = duplicate_group.ne("EXCLUDED").astype("int8")

    count_sources = {
        "i_prior_duplicate_candidate_count": "rule_prior_duplicate_candidate_count",
        "i_prior_mutex_overlap_candidate_count": "rule_prior_mutex_overlap_candidate_count",
        "i_reference_match_count": "rule_reference_match_count",
        "i_prior_promotion_candidate_count": "rule_prior_promotion_candidate_count",
        "i_prior_optcpc_conflict_candidate_count": "rule_prior_optcpc_conflict_candidate_count",
        "i_prior_pay_fix_event_candidate_count": "rule_prior_pay_fix_event_candidate_count",
        "i_prior_family_member_count": "rule_prior_family_member_count",
        "i_prior_missing_relationship_count": "rule_prior_missing_relationship_count",
    }
    for feature_name, source_name in count_sources.items():
        features[feature_name] = _numeric_source(frame, source_name).fillna(0)

    features["i_has_prior_duplicate_candidate"] = features["i_prior_duplicate_candidate_count"].gt(0).astype("int8")
    features["i_description_requires_mutex_check"] = description.isin(
        MUTEX_DESCRIPTION_SET
    ).astype("int8")
    features["i_has_prior_mutex_overlap_candidate"] = (
        features["i_prior_mutex_overlap_candidate_count"].gt(0).astype("int8")
    )

    reference_text = (
        frame.get("ref_do2_no", pd.Series("", index=frame.index))
        .fillna("")
        .astype("string")
        .str.strip()
    )
    features["i_reference_present"] = reference_text.ne("").astype("int8")
    features["i_reference_format_valid"] = (
        (
            reference_text.str.fullmatch(r"[^/]+/\d{4}/\d{3}/\d{4}", na=False)
        ).astype("int8")
    )
    features["i_reference_found"] = features["i_reference_match_count"].gt(0).astype("int8")

    fs_available = from_date.notna() & employee_fs_due_date.notna()
    features["i_from_date_within_fs_month"] = _rule_bool(
        from_date.dt.to_period("M").eq(employee_fs_due_date.dt.to_period("M")),
        fs_available,
    )
    features["i_is_optfix_or_profix"] = description.isin(
        {"OPTFIX", "PROFIX"}
    ).astype("int8")
    features["i_pay_fixation_candidate_support_missing"] = (
        features["i_is_optfix_or_profix"].eq(1)
        & features["i_prior_promotion_candidate_count"].eq(0)
    ).astype("int8")
    features["i_has_prior_optcpc_conflict_candidate"] = (
        features["i_prior_optcpc_conflict_candidate_count"].gt(0).astype("int8")
    )

    # Appendix-J required-field checks use only entry-time DO2 values.
    appendix_rule_found = (
        _numeric_source(frame, "rule_appendix_j_id").notna()
        & USE_CURRENT_APPENDIX_J_FOR_HISTORICAL_ROWS
    )
    required_checks = {
        "from_date": "rule_appendix_from_date_req",
        "to_date": "rule_appendix_to_date_req",
        "rate": "rule_appendix_rate_req",
        "tjra": "rule_appendix_tjra_req",
        "lra": "rule_appendix_lra_req",
        "ref_do2_no": "rule_appendix_ref_do2_no_req",
    }
    appendix_failures = pd.DataFrame(index=frame.index)
    for source_column, rule_column in required_checks.items():
        required = (
            _truthy_source(frame, rule_column)
            & USE_CURRENT_APPENDIX_J_FOR_HISTORICAL_ROWS
        )
        missing = _column_present(frame, source_column).eq(0)
        appendix_failures[source_column] = required & missing
        features[f"i_appendix_required_{source_column}_missing"] = (
            required & missing
        ).astype("int8")
    features["i_appendix_j_rule_found"] = appendix_rule_found.astype("int8")
    features["i_appendix_required_missing_count"] = (
        appendix_failures.sum(axis=1).astype("float32")
    )
    features["i_relationship_value_missing"] = _column_present(
        frame, "column_b"
    ).eq(0).astype("int8")

    features["i_service_days_at_from_date"] = (
        from_date - enrolment_date
    ).dt.days.astype("float32")
    features["i_age_days_at_from_date"] = (
        from_date - birth_date
    ).dt.days.astype("float32")
    features["i_less_than_six_months_service"] = _rule_bool(
        features["i_service_days_at_from_date"].lt(183),
        features["i_service_days_at_from_date"].notna(),
    )
    features["i_less_than_eight_years_service"] = _rule_bool(
        features["i_service_days_at_from_date"].lt(8 * 365.25),
        features["i_service_days_at_from_date"].notna(),
    )

    features["i_rule_history_cutoff_missing"] = _numeric_source(
        frame, "rule_history_cutoff_available"
    ).fillna(0).eq(0).astype("int8")
    return features


def _normalized_rule_text(frame, column):
    if column not in frame:
        return pd.Series("", index=frame.index, dtype="string")
    return (
        frame[column]
        .fillna("")
        .astype("string")
        .str.strip()
        .str.upper()
    )


def build_status_mr_features(frame):
    """Manual-rejection facts confirmed by Java/application evidence."""
    features = pd.DataFrame(index=frame.index)
    appendix_rule_found = (
        _numeric_source(frame, "rule_appendix_j_id").notna()
        & USE_CURRENT_APPENDIX_J_FOR_HISTORICAL_ROWS
    )
    appendix_exact_match = _numeric_source(
        frame, "rule_appendix_exact_value_match"
    ).fillna(0).gt(0)
    features["mr_entry_flag"] = _truthy_source(frame, "mr").astype("int8")
    features["mr_appendix_value_combination_invalid"] = (
        appendix_rule_found & ~appendix_exact_match
    ).astype("int8")
    return features


def build_status_v_features(frame):
    """Verified static acceptance evidence; no outcome or workflow columns."""
    features = pd.DataFrame(index=frame.index)
    description = _clean_category(
        frame.get("description", pd.Series("", index=frame.index))
    )
    from_date = _date(frame, "from_date")
    birth_date = _date(frame, "employee_birth_date")
    transcription_type = _normalized_text(frame, "transcription_type")
    column_a = _normalized_text(frame, "column_a")
    cilq_granted = description.eq("CILQ") & column_a.eq("GRANTED")
    cilq_age_days = (from_date - birth_date).dt.days
    features["v_appt_nonactionable_after_7cpc"] = (
        description.eq("APPT") & from_date.ge(pd.Timestamp("2017-07-01"))
    ).astype("int8")

    appendix_rule_found = (
        _numeric_source(frame, "rule_appendix_j_id").notna()
        & USE_CURRENT_APPENDIX_J_FOR_HISTORICAL_ROWS
    )
    appendix_exact_match = _numeric_source(
        frame, "rule_appendix_exact_value_match"
    ).fillna(0).gt(0)
    static_failures = pd.DataFrame(index=frame.index)
    static_failures["abolished"] = (
        description.isin(ABOLISHED_DESCRIPTION_SET)
        & from_date.ge(pd.Timestamp("2017-07-01"))
    )
    static_failures["army_no_missing_at_save"] = (
        _column_present(frame, "army_no").eq(0)
    )
    static_failures["description_missing_at_save"] = (
        _column_present(frame, "description").eq(0)
    )
    static_failures["daa_inadmissible"] = description.eq("DAA")
    static_failures["bunch_wrong_source"] = (
        description.eq("BUNCH") & transcription_type.ne("T")
    )
    static_failures["cilq_birth_date_missing"] = (
        cilq_granted & birth_date.isna()
    )
    static_failures["cilq_from_date_missing"] = (
        cilq_granted & from_date.isna()
    )
    static_failures["cilq_age_below_25"] = (
        cilq_granted & birth_date.notna() & from_date.notna()
        & cilq_age_days.lt(25 * 365.2425)
    )
    static_failures["negative_duration"] = (
        _date(frame, "from_date").notna()
        & _date(frame, "to_date").notna()
        & _date(frame, "from_date").gt(_date(frame, "to_date"))
    )
    static_failures["appendix_combination"] = appendix_rule_found & ~appendix_exact_match
    features["v_verified_static_violation_count"] = (
        static_failures.sum(axis=1).astype("float32")
    )
    features["v_all_verified_static_checks_passed"] = (
        features["v_verified_static_violation_count"].eq(0).astype("int8")
    )
    return features


def build_features(
    frame: pd.DataFrame,
    *,
    expected_columns: list[str] | None = None,
    expected_categorical: list[str] | None = None,
    expected_text: list[str] | None = None,
) -> FeatureResult:
    """Build the same compact feature matrix for every data split."""
    del expected_text  # Native CatBoost text handling is intentionally disabled.
    result = pd.DataFrame(index=frame.index)
    categorical: list[str] = []

    configured = set(CATEGORICAL_BASE) | set(NUMERIC_BASE)
    leaking_features = sorted(configured & set(leakage_columns_present(frame.columns)))
    if leaking_features:
        raise RuntimeError(f"Leakage columns configured as model features: {leaking_features}")

    for column in CATEGORICAL_BASE:
        if column in frame:
            result[column] = _clean_category(frame[column])
            categorical.append(column)

    for column in NUMERIC_BASE:
        value = pd.to_numeric(_series(frame, column), errors="coerce")
        result[column] = value.astype("float32")
        result[f"{column}_is_missing"] = value.isna().astype("int8")
        result[f"{column}_is_zero"] = value.eq(0).fillna(False).astype("int8")

    do2_date = _date(frame, "do2_date")
    from_date = _date(frame, "from_date")
    to_date = _date(frame, "to_date")
    created_at = _date(frame, "created_at")

    result["duration_days"] = (to_date - from_date).dt.days.astype("float32")
    raw_submission_delay_days = (
        created_at.dt.normalize() - do2_date.dt.normalize()
    ).dt.days.astype("float32")

    # Delay is a genuine bill property, not a calendar feature. Extreme values
    # are capped so a temporary migration/backlog cannot dominate tree splits.
    result["submission_delay_days_capped"] = raw_submission_delay_days.clip(
        lower=-30, upper=730
    ).astype("float32")
    result["submission_delay_over_730_days"] = (
        raw_submission_delay_days.gt(730).fillna(False).astype("int8")
    )
    result["negative_duration"] = (
        from_date.notna() & to_date.notna() & from_date.gt(to_date)
    ).astype("int8")
    result["from_date_after_do2_date"] = (
        from_date.notna() & do2_date.notna()
        & from_date.dt.normalize().gt(do2_date.dt.normalize())
    ).astype("int8")

    delay_bucket = pd.cut(
        raw_submission_delay_days,
        bins=[-np.inf, -1, 30, 90, 365, np.inf],
        labels=[
            "FUTURE_DO2_DATE", "D00_030", "D031_090",
            "D091_365", "D366_PLUS",
        ],
    ).astype("string").fillna("MISSING").astype(str)
    result["pattern_delay_bucket"] = delay_bucket

    # Bill-pattern categories contain no status, reason, approval, month,
    # year, or aggregate status percentage. Values are normalized and bounded
    # to prevent unbounded free-text categories.
    description_pattern = _clean_category(_series(frame, "description"))
    group_pattern = _clean_category(_series(frame, "do2_group"))
    type_pattern = _clean_category(_series(frame, "do2_type"))
    source_pattern = _clean_category(_series(frame, "transcription_type"))
    column_a_pattern = _clean_category(_series(frame, "column_a")).str.slice(0, 64)
    column_b_pattern = _clean_category(_series(frame, "column_b")).str.slice(0, 64)
    column_c_pattern = _clean_category(_series(frame, "column_c")).str.slice(0, 64)

    has_a = _column_present(frame, "column_a").astype(str)
    has_b = _column_present(frame, "column_b").astype(str)
    has_c = _column_present(frame, "column_c").astype(str)
    column_presence = "A" + has_a + "B" + has_b + "C" + has_c

    result["pattern_column_a_value"] = column_a_pattern
    result["pattern_column_b_value"] = column_b_pattern
    result["pattern_column_c_value"] = column_c_pattern
    result["pattern_description_group"] = description_pattern + "|" + group_pattern
    result["pattern_description_type"] = description_pattern + "|" + type_pattern
    result["pattern_description_source"] = description_pattern + "|" + source_pattern
    result["pattern_description_delay"] = description_pattern + "|" + delay_bucket
    result["pattern_description_columns"] = description_pattern + "|" + column_presence

    # Stable bill identity deliberately excludes source and delay. Those
    # operational properties remain separate candidates and must prove stable
    # across temporal folds before the selector may retain them.
    result["pattern_bill_signature"] = (
        description_pattern
        + "|" + group_pattern
        + "|" + type_pattern
        + "|" + column_presence
    )
    result["pattern_operational_signature"] = (
        result["pattern_bill_signature"]
        + "|" + source_pattern
        + "|" + delay_bucket
    )

    pattern_categorical_columns = [
        "pattern_delay_bucket",
        "pattern_column_a_value",
        "pattern_column_b_value",
        "pattern_column_c_value",
        "pattern_description_group",
        "pattern_description_type",
        "pattern_description_source",
        "pattern_description_delay",
        "pattern_description_columns",
        "pattern_bill_signature",
        "pattern_operational_signature",
    ]
    categorical.extend(pattern_categorical_columns)

    for feature_name, source_column in {
        "has_reference_do2": "ref_do2_no",
        "has_attachment": "attachment",
        "has_column_a": "column_a",
        "has_column_b": "column_b",
        "has_column_c": "column_c",
    }.items():
        result[feature_name] = _column_present(frame, source_column)

    core_presence = pd.DataFrame(
        {column: _column_present(frame, column) for column in CORE_REQUIRED_COLUMNS},
        index=frame.index,
    )
    result["missing_required_field_count"] = (
        len(CORE_REQUIRED_COLUMNS) - core_presence.sum(axis=1)
    ).astype("float32")

    result = pd.concat(
        [
            result,
            build_status_n_features(frame),
            build_status_i_features(frame),
            build_status_mr_features(frame),
            build_status_v_features(frame),
        ],
        axis=1,
    )

    # The duplicate-rule group is intentionally categorical.
    for rule_categorical in ("i_duplicate_rule_group",):
        if rule_categorical in result:
            result[rule_categorical] = _clean_category(result[rule_categorical])
            if rule_categorical not in categorical:
                categorical.append(rule_categorical)

    if result.columns.duplicated().any():
        duplicates = result.columns[result.columns.duplicated()].tolist()
        raise RuntimeError(f"Duplicate feature names: {duplicates}")

    if expected_columns is not None:
        expected_categorical_set = set(expected_categorical or [])
        for column in expected_columns:
            if column not in result:
                result[column] = (
                    MISSING_CATEGORY if column in expected_categorical_set else np.nan
                )
        result = result[expected_columns]
        categorical = [
            column for column in expected_columns if column in expected_categorical_set
        ]
    else:
        categorical = [column for column in categorical if column in result]

    if result.empty:
        raise RuntimeError("No model features could be built from the supplied columns.")

    return FeatureResult(
        frame=result,
        categorical_columns=categorical,
        text_columns=[],
    )


## 7. Build aligned training and validation matrices

In [9]:
from sklearn.preprocessing import LabelEncoder

# Keep status separately as the correct answer (y).
y_train_status = normalize_status(train_df["status"])
y_validation_status = normalize_status(validation_df["status"])

# Remove status before feature engineering. The model never receives status.
X_train_raw = train_df.drop(
    columns=["status", "target", "is_non_valid"],
    errors="ignore",
).copy()
X_validation_raw = validation_df.drop(
    columns=["status", "target", "is_non_valid"],
    errors="ignore",
).copy()

train_features = build_features(X_train_raw)

# Remove columns that cannot teach the model anything in this training split.
# Validation and test are then aligned to the retained training columns.
UNINFORMATIVE_TRAIN_FEATURES = [
    column
    for column in train_features.frame.columns
    if train_features.frame[column].nunique(dropna=False) <= 1
]
if UNINFORMATIVE_TRAIN_FEATURES:
    train_features = FeatureResult(
        frame=train_features.frame.drop(columns=UNINFORMATIVE_TRAIN_FEATURES),
        categorical_columns=[
            column for column in train_features.categorical_columns
            if column not in UNINFORMATIVE_TRAIN_FEATURES
        ],
        text_columns=[
            column for column in train_features.text_columns
            if column not in UNINFORMATIVE_TRAIN_FEATURES
        ],
    )

validation_features = build_features(
    X_validation_raw,
    expected_columns=list(train_features.frame.columns),
    expected_categorical=train_features.categorical_columns,
    expected_text=train_features.text_columns,
)

FEATURE_COLUMNS = list(train_features.frame.columns)
CATEGORICAL_COLUMNS = list(train_features.categorical_columns)
TEXT_COLUMNS = list(train_features.text_columns)

status_encoder = LabelEncoder()
y_train_encoded = status_encoder.fit_transform(y_train_status)
STATUS_CLASSES = status_encoder.classes_.astype(str).tolist()

missing_training_classes = sorted(set(VALID_STATUSES) - set(STATUS_CLASSES))
if missing_training_classes:
    raise ValueError(
        "Training data is missing required status classes: "
        + ", ".join(missing_training_classes)
    )

validation_seen_mask = y_validation_status.isin(STATUS_CLASSES).to_numpy()
unseen_validation_classes = sorted(
    set(y_validation_status) - set(STATUS_CLASSES)
)

print("Feature count:", len(FEATURE_COLUMNS))
print("Categorical features:", len(CATEGORICAL_COLUMNS))
print("Text features:", len(TEXT_COLUMNS))
print("Training classes:", STATUS_CLASSES)
print("Unseen validation classes:", unseen_validation_classes)
print("Status present in training features:", "status" in train_features.frame.columns)
print("Status present in validation features:", "status" in validation_features.frame.columns)


print("Automatically removed constant/all-missing features:", UNINFORMATIVE_TRAIN_FEATURES)
rule_feature_columns = [
    column for column in FEATURE_COLUMNS
    if column.startswith(("n_", "i_", "mr_", "v_"))
]
print("Verified N/I/MR/V rule feature count:", len(rule_feature_columns))


# ------------------------------------------------------------------
# Calendar-feature guard: date windows may select rows, but calendar workload
# or target-distribution signals must never enter the model.
# ------------------------------------------------------------------
FORBIDDEN_CALENDAR_FEATURES = frozenset({
    "created_month", "created_year", "created_day", "created_day_of_week",
    "do2_month", "do2_year", "do2_day", "do2_day_of_week",
    "monthly_bill_count", "monthly_status_percentage",
    "monthly_invalid_count", "monthly_valid_count",
    "i_days_after_7cpc_abolition",
})
forbidden_calendar_features_present = sorted(
    set(FEATURE_COLUMNS) & FORBIDDEN_CALENDAR_FEATURES
)
if forbidden_calendar_features_present:
    raise RuntimeError(
        "Forbidden calendar/workload model features found: "
        + ", ".join(forbidden_calendar_features_present)
    )

PATTERN_KEY_COLUMN = "pattern_bill_signature"
if PATTERN_KEY_COLUMN not in FEATURE_COLUMNS:
    raise RuntimeError(f"Required pattern key is missing: {PATTERN_KEY_COLUMN}")

TRAIN_PATTERN_KEYS = frozenset(
    train_features.frame[PATTERN_KEY_COLUMN].astype(str).unique()
)

def build_pattern_purity(pattern_keys, statuses):
    source = pd.DataFrame({
        "bill_pattern_key": pd.Series(pattern_keys).astype(str).to_numpy(),
        "status": pd.Series(statuses).astype(str).to_numpy(),
    })
    counts = pd.crosstab(source["bill_pattern_key"], source["status"])
    for status_class in VALID_STATUSES:
        if status_class not in counts:
            counts[status_class] = 0
    counts = counts[list(VALID_STATUSES)]
    report = counts.reset_index()
    report["rows"] = counts.sum(axis=1).to_numpy()
    report["dominant_status"] = counts.idxmax(axis=1).to_numpy()
    report["dominant_status_rows"] = counts.max(axis=1).to_numpy()
    report["pattern_purity"] = (
        report["dominant_status_rows"] / report["rows"]
    )
    return report.sort_values(
        ["rows", "pattern_purity"], ascending=[False, False]
    ).reset_index(drop=True)


pattern_purity_report = build_pattern_purity(
    train_features.frame[PATTERN_KEY_COLUMN], y_train_status
)
print("Forbidden calendar features present:", forbidden_calendar_features_present)
print("Unique training bill patterns:", len(TRAIN_PATTERN_KEYS))
display(pattern_purity_report.head(30))


Feature count: 90
Categorical features: 18
Text features: 0
Training classes: ['I', 'MR', 'N', 'V']
Unseen validation classes: []
Status present in training features: False
Status present in validation features: False
Automatically removed constant/all-missing features: ['n_lpc_first_row_nondeterministic', 'i_army_no_missing_at_save', 'i_description_missing_at_save', 'i_daa_always_inadmissible', 'i_bunch_wrong_transcription_type', 'i_cilq_granted_birth_date_missing', 'i_cilq_granted_from_date_missing', 'i_prior_missing_relationship_count', 'i_appendix_required_from_date_missing', 'i_appendix_required_to_date_missing', 'i_appendix_required_rate_missing', 'i_appendix_required_tjra_missing', 'i_appendix_required_lra_missing', 'i_appendix_required_ref_do2_no_missing', 'i_appendix_j_rule_found', 'i_appendix_required_missing_count', 'i_rule_history_cutoff_missing', 'mr_entry_flag', 'mr_appendix_value_combination_invalid']
Verified N/I/MR/V rule feature count: 53
Forbidden calendar features p

status,bill_pattern_key,I,MR,N,V,rows,dominant_status,dominant_status_rows,pattern_purity
0,CANCEL|IA|0|A1B0C0,2435,6323,1174,24214,34146,V,24214,0.709131
1,INCR|IA|0|A0B0C1,20619,1,87,7774,28481,I,20619,0.723956
2,HRA|IA|0|A1B0C0,18404,183,70,3774,22431,I,18404,0.820472
3,PAL|IA|0|A0B0C0,432,71,84,15796,16383,V,15796,0.964170
4,DA|IA|0|A0B0C0,9,1,0,15767,15777,V,15767,0.999366
5,RISK|IA|0|A1B1C1,827,13,1386,9823,12049,V,9823,0.815254
6,ACCLVE|IIB|0|A1B1C1,1093,4609,196,6057,11955,V,6057,0.506650
7,CL|IA|0|A0B0C0,269,81,80,11315,11745,V,11315,0.963389
8,HAUCA|IA|0|A1B1C1,3871,12,94,5998,9975,V,5998,0.601303
9,RISK|IA|0|A1B0C0,123,11,0,8881,9015,V,8881,0.985136


In [10]:
# Show every final model feature and how CatBoost will receive it.
raw_input_columns = set(X_train_raw.columns)

feature_inventory = pd.DataFrame({
    "feature_number": range(1, len(FEATURE_COLUMNS) + 1),
    "feature": FEATURE_COLUMNS,
    "origin": [
        "direct/raw" if column in raw_input_columns else "engineered"
        for column in FEATURE_COLUMNS
    ],
    "model_type": [
        "categorical" if column in CATEGORICAL_COLUMNS
        else "text" if column in TEXT_COLUMNS
        else "numeric"
        for column in FEATURE_COLUMNS
    ],
})

display(feature_inventory)
print("Final feature count:", len(FEATURE_COLUMNS))
print("Features intentionally removed:", sorted(FEATURES_TO_DROP))

print("Automatically removed as uninformative:", UNINFORMATIVE_TRAIN_FEATURES)


,feature_number,feature,origin,model_type
0,1,do2_group,direct/raw,categorical
1,2,do2_type,direct/raw,categorical
2,3,description,direct/raw,categorical
3,4,transcription_type,direct/raw,categorical
4,5,delayed,direct/raw,categorical
...,...,...,...,...
85,86,i_less_than_six_months_service,engineered,numeric
86,87,i_less_than_eight_years_service,engineered,numeric
87,88,v_appt_nonactionable_after_7cpc,engineered,numeric
88,89,v_verified_static_violation_count,engineered,numeric


Final feature count: 90
Features intentionally removed: ['abs_duration_days', 'abs_submission_delay_days', 'army_no_is_placeholder', 'check_digit', 'conditional_required_missing_count', 'conditional_required_present_count', 'core_required_present_count', 'created_at_plausible', 'created_before_do2', 'description_char_count', 'description_do2_type', 'description_has_digit', 'description_word_count', 'do2_date_plausible', 'do2_group_type', 'do2_item_no', 'from_date_plausible', 'has_cfa_sanction_date', 'has_cfa_sanction_letter_no', 'has_cfa_sanction_received', 'has_column9_remarks', 'has_pto_remarks', 'has_punishment', 'log_abs_duration_days', 'log_abs_submission_delay_days', 'long_duration', 'lra_is_negative', 'lra_log_abs', 'prior_army_do2_type_row_count', 'prior_batch_do2_item_share', 'prior_batch_row_count', 'prior_do2_no_row_count', 'prior_unit_do2_type_row_count', 'punishment', 'qe', 'rate_is_negative', 'rate_log_abs', 'rate_lra_abs_diff', 'rate_tjra_abs_diff', 'same_day_created_do2

## 8. Train CatBoost with training-period prior correction

In [11]:
import pandas as pd
from pandas.api.types import is_object_dtype, is_string_dtype
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)


def is_string_like(series):
    return (
        is_object_dtype(series.dtype)
        or is_string_dtype(series.dtype)
        or isinstance(series.dtype, pd.CategoricalDtype)
    )


detected_categorical = [
    column
    for column in train_features.frame.columns
    if (
        is_string_like(train_features.frame[column])
        or is_string_like(validation_features.frame[column])
    )
]
CATEGORICAL_COLUMNS = [
    column
    for column in FEATURE_COLUMNS
    if column in set(CATEGORICAL_COLUMNS) | set(detected_categorical)
]
TEXT_COLUMNS = []
NUMERIC_COLUMNS = [
    column for column in FEATURE_COLUMNS if column not in CATEGORICAL_COLUMNS
]


def prepare_model_frame(feature_frame):
    prepared = feature_frame.reindex(columns=FEATURE_COLUMNS).copy()
    for column in CATEGORICAL_COLUMNS:
        prepared[column] = (
            prepared[column]
            .astype("string")
            .fillna(MISSING_CATEGORY)
            .str.strip()
            .replace("", MISSING_CATEGORY)
            .astype(str)
        )
    for column in NUMERIC_COLUMNS:
        prepared[column] = pd.to_numeric(prepared[column], errors="coerce")
    return prepared


X_train_model = prepare_model_frame(train_features.frame)
X_validation_model = prepare_model_frame(validation_features.frame)
y_validation_encoded = status_encoder.transform(y_validation_status)

if list(X_train_model.columns) != list(X_validation_model.columns):
    raise ValueError("Training and validation feature columns do not match")

print("Training shape:", X_train_model.shape)
print("Validation shape:", X_validation_model.shape)
print("Categorical feature count:", len(CATEGORICAL_COLUMNS))
print("Numeric feature count:", len(NUMERIC_COLUMNS))
print("Status in model input:", "status" in X_train_model.columns)


# Correct only the deliberate V sampling. No month, year, or calendar status
# percentage is used as a feature or as a hand-written class weight.
sampled_training_prior = (
    y_train_status.value_counts(normalize=True)
    .reindex(STATUS_CLASSES)
    .fillna(0.0)
)
expected_real_prior = (
    training_population_prior.set_index("status")["rows"]
    .reindex(STATUS_CLASSES)
    .fillna(0.0)
    .astype(float)
)
expected_real_prior = expected_real_prior / expected_real_prior.sum()

if sampled_training_prior.eq(0).any():
    missing = sampled_training_prior[sampled_training_prior.eq(0)].index.tolist()
    raise ValueError(f"Sampled training data is missing classes: {missing}")


def make_sample_weight(statuses, power):
    statuses = pd.Series(statuses).reset_index(drop=True)
    sampled_prior = (
        statuses.value_counts(normalize=True)
        .reindex(STATUS_CLASSES)
        .fillna(0.0)
    )
    missing = sampled_prior[sampled_prior.eq(0)].index.tolist()
    if missing:
        raise ValueError(f"Training fold is missing classes: {missing}")
    weight_by_status = (expected_real_prior / sampled_prior) ** float(power)
    weights = statuses.map(weight_by_status).astype(float).to_numpy()
    return weights / weights.mean(), weight_by_status


def temporal_macro_summary(actual, predicted, timestamps):
    """Report month stability; month is never supplied to CatBoost."""
    scored = pd.DataFrame({
        "actual": np.asarray(actual),
        "predicted": np.asarray(predicted),
        "month": parse_datetime_mixed(pd.Series(timestamps))
        .dt.to_period("M")
        .astype(str),
    })
    rows = []
    for month, group in scored.groupby("month", dropna=False):
        rows.append({
            "month": str(month),
            "rows": int(len(group)),
            "macro_f1": float(f1_score(
                group["actual"],
                group["predicted"],
                labels=STATUS_CLASSES,
                average="macro",
                zero_division=0,
            )),
        })
    monthly = pd.DataFrame(rows)
    return {
        "mean_monthly_macro_f1": float(monthly["macro_f1"].mean()),
        "worst_monthly_macro_f1": float(monthly["macro_f1"].min()),
        "monthly": monthly,
    }


def equal_weight_monthly_macro_f1(actual, predicted, timestamps):
    summary = temporal_macro_summary(actual, predicted, timestamps)
    return summary["mean_monthly_macro_f1"], summary["monthly"]


BASE_MODEL_PARAMETERS = {
    "learning_rate": LEARNING_RATE,
    "depth": DEPTH,
    "l2_leaf_reg": 10,
    "min_data_in_leaf": 100,
    "loss_function": "MultiClass",
    "eval_metric": "MultiClass",
    "custom_metric": [
        "Accuracy", "TotalF1:average=Macro", "TotalF1:average=Weighted",
    ],
    "random_seed": RANDOM_SEED,
    "task_type": TASK_TYPE,
    "allow_writing_files": False,
    "verbose": False,
    "od_type": "Iter",
}
if TASK_TYPE == "GPU":
    BASE_MODEL_PARAMETERS["devices"] = DEVICES


def make_pool(frame, labels, columns, categorical_columns, *, weight=None):
    return Pool(
        data=frame[columns],
        label=labels,
        cat_features=[c for c in categorical_columns if c in columns],
        weight=weight,
    )


# ---------------------------------------------------------
# 1. Select the V-sampling correction by worst-month performance first.
# ---------------------------------------------------------
rule_prefixes = ("n_", "i_", "mr_", "v_")
rule_columns = [c for c in FEATURE_COLUMNS if c.startswith(rule_prefixes)]
stable_pattern_columns = [
    c for c in (
        "pattern_column_a_value",
        "pattern_column_b_value",
        "pattern_column_c_value",
        "pattern_description_group",
        "pattern_description_type",
        "pattern_description_columns",
        "pattern_bill_signature",
    )
    if c in FEATURE_COLUMNS
]
operational_pattern_columns = [
    c for c in (
        "pattern_delay_bucket",
        "pattern_description_source",
        "pattern_description_delay",
        "pattern_operational_signature",
    )
    if c in FEATURE_COLUMNS
]
all_pattern_columns = [c for c in FEATURE_COLUMNS if c.startswith("pattern_")]
base_columns = [
    c for c in FEATURE_COLUMNS
    if c not in set(all_pattern_columns) and c not in set(rule_columns)
]

prior_sweep_columns = list(dict.fromkeys(
    base_columns
    + rule_columns
    + [c for c in ("submission_delay_days_capped",) if c in FEATURE_COLUMNS]
))
prior_sweep_categorical = [
    c for c in CATEGORICAL_COLUMNS if c in prior_sweep_columns
]

SWEEP_ITERATIONS = min(ITERATIONS, 600)
all_validation_pool = make_pool(
    X_validation_model,
    y_validation_encoded,
    prior_sweep_columns,
    prior_sweep_categorical,
)
sweep_rows = []

for power in PRIOR_CORRECTION_POWER_GRID:
    weight, _ = make_sample_weight(y_train_status, power)
    sweep_pool = make_pool(
        X_train_model,
        y_train_encoded,
        prior_sweep_columns,
        prior_sweep_categorical,
        weight=weight,
    )
    sweep_model = CatBoostClassifier(
        **dict(BASE_MODEL_PARAMETERS, iterations=SWEEP_ITERATIONS)
    )
    sweep_model.fit(
        sweep_pool,
        eval_set=all_validation_pool,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        use_best_model=True,
    )
    probabilities = sweep_model.predict_proba(
        X_validation_model[prior_sweep_columns]
    )
    predicted = status_encoder.inverse_transform(probabilities.argmax(axis=1))
    overall_macro = f1_score(
        y_validation_status,
        predicted,
        labels=STATUS_CLASSES,
        average="macro",
        zero_division=0,
    )
    temporal = temporal_macro_summary(
        y_validation_status,
        predicted,
        validation_df[SPLIT_DATE_COLUMN],
    )
    sweep_rows.append({
        "prior_correction_power": float(power),
        "validation_worst_monthly_macro_f1": temporal["worst_monthly_macro_f1"],
        "validation_mean_monthly_macro_f1": temporal["mean_monthly_macro_f1"],
        "validation_macro_f1": float(overall_macro),
        "validation_accuracy": float((predicted == y_validation_status).mean()),
        "best_iteration": int(sweep_model.get_best_iteration()),
    })
    print(
        f"power={power:.2f} "
        f"worst_month={temporal['worst_monthly_macro_f1']:.4f} "
        f"mean_month={temporal['mean_monthly_macro_f1']:.4f} "
        f"macro_f1={overall_macro:.4f}"
    )
    del sweep_model, sweep_pool

prior_correction_sweep = pd.DataFrame(sweep_rows).sort_values(
    [
        "validation_worst_monthly_macro_f1",
        "validation_mean_monthly_macro_f1",
        "validation_macro_f1",
    ],
    ascending=False,
).reset_index(drop=True)
display(prior_correction_sweep)

BEST_PRIOR_CORRECTION_POWER = float(
    prior_correction_sweep.iloc[0]["prior_correction_power"]
)
train_sample_weight, sampling_weight_by_status = make_sample_weight(
    y_train_status, BEST_PRIOR_CORRECTION_POWER
)
sampling_weight_report = pd.DataFrame({
    "sampled_training_prior": sampled_training_prior,
    "natural_training_prior": expected_real_prior,
    "training_weight": sampling_weight_by_status,
})
print("Selected PRIOR_CORRECTION_POWER:", BEST_PRIOR_CORRECTION_POWER)
display(sampling_weight_report)


# ---------------------------------------------------------
# 2. Select the feature group on chronological stability folds.
# ---------------------------------------------------------
delay_numeric_columns = [
    c for c in (
        "submission_delay_days_capped",
        "submission_delay_over_730_days",
    )
    if c in FEATURE_COLUMNS
]
base_without_source = [c for c in base_columns if c != "transcription_type"]

ablation_groups = {
    "RULES_NO_PATTERN_NO_DELAY": list(dict.fromkeys(
        base_columns + rule_columns
    )),
    "RULES_NO_PATTERN_CAPPED_DELAY": list(dict.fromkeys(
        base_columns + rule_columns + delay_numeric_columns
    )),
    "RULES_NO_PATTERN_NO_SOURCE": list(dict.fromkeys(
        base_without_source + rule_columns + delay_numeric_columns
    )),
}


development_X = pd.concat(
    [X_train_model, X_validation_model],
    ignore_index=True,
)
development_y_status = pd.concat(
    [y_train_status.reset_index(drop=True), y_validation_status.reset_index(drop=True)],
    ignore_index=True,
)
development_timestamps = pd.concat(
    [
        parse_datetime_mixed(train_df[SPLIT_DATE_COLUMN]).reset_index(drop=True),
        parse_datetime_mixed(validation_df[SPLIT_DATE_COLUMN]).reset_index(drop=True),
    ],
    ignore_index=True,
)


def temporal_fold_masks(validation_start, validation_end):
    start = pd.Timestamp(validation_start)
    end = pd.Timestamp(validation_end)
    train_mask = (
        development_timestamps.ge(pd.Timestamp(TRAIN_START))
        & development_timestamps.lt(start)
    ).to_numpy()
    validation_mask = (
        development_timestamps.ge(start)
        & development_timestamps.lt(end)
    ).to_numpy()
    return train_mask, validation_mask


feature_fold_rows = []
if RUN_FEATURE_GROUP_ABLATION:
    for group_name, group_columns in ablation_groups.items():
        group_categorical = [
            c for c in CATEGORICAL_COLUMNS if c in group_columns
        ]
        for fold_name, fold_start, fold_end in STABILITY_FOLDS:
            fold_train_mask, fold_validation_mask = temporal_fold_masks(
                fold_start, fold_end
            )
            fold_train_rows = int(fold_train_mask.sum())
            fold_validation_rows = int(fold_validation_mask.sum())
            if min(fold_train_rows, fold_validation_rows) < MIN_STABILITY_FOLD_ROWS:
                raise ValueError(
                    f"Temporal fold {fold_name} is too small: "
                    f"train={fold_train_rows:,}, validation={fold_validation_rows:,}"
                )

            fold_y_train = development_y_status.loc[fold_train_mask].reset_index(drop=True)
            fold_y_validation = development_y_status.loc[
                fold_validation_mask
            ].reset_index(drop=True)
            missing_train = sorted(set(STATUS_CLASSES) - set(fold_y_train))
            missing_validation = sorted(set(STATUS_CLASSES) - set(fold_y_validation))
            if missing_train or missing_validation:
                raise ValueError(
                    f"Fold {fold_name} class coverage failure. "
                    f"missing_train={missing_train}, "
                    f"missing_validation={missing_validation}"
                )

            fold_train_weight, _ = make_sample_weight(
                fold_y_train, BEST_PRIOR_CORRECTION_POWER
            )
            # The 2024 validation folds come from the deliberately V-sampled
            # development extract. Reconstruct their natural class prior for
            # scoring only. The natural 2025 validation fold needs no change.
            if pd.Timestamp(fold_end) <= pd.Timestamp(TRAIN_END):
                fold_metric_weight, _ = make_sample_weight(
                    fold_y_validation, 1.0
                )
                validation_prior_reconstructed = True
            else:
                fold_metric_weight = np.ones(
                    len(fold_y_validation), dtype=float
                )
                validation_prior_reconstructed = False
            fold_train_pool = make_pool(
                development_X.loc[fold_train_mask].reset_index(drop=True),
                status_encoder.transform(fold_y_train),
                group_columns,
                group_categorical,
                weight=fold_train_weight,
            )
            fold_validation_pool = make_pool(
                development_X.loc[fold_validation_mask].reset_index(drop=True),
                status_encoder.transform(fold_y_validation),
                group_columns,
                group_categorical,
            )
            fold_model = CatBoostClassifier(
                **dict(
                    BASE_MODEL_PARAMETERS,
                    iterations=min(ITERATIONS, ABLATION_ITERATIONS),
                )
            )
            fold_model.fit(
                fold_train_pool,
                eval_set=fold_validation_pool,
                early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                use_best_model=True,
            )
            fold_probabilities = fold_model.predict_proba(
                development_X.loc[fold_validation_mask, group_columns]
            )
            fold_predicted = status_encoder.inverse_transform(
                fold_probabilities.argmax(axis=1)
            )
            precision, recall, class_f1, support = (
                precision_recall_fscore_support(
                    fold_y_validation,
                    fold_predicted,
                    labels=STATUS_CLASSES,
                    zero_division=0,
                    sample_weight=fold_metric_weight,
                )
            )
            actual_non_v = fold_y_validation.ne("V").to_numpy()
            predicted_non_v = fold_predicted != "V"
            feature_fold_rows.append({
                "feature_group": group_name,
                "feature_count": int(len(group_columns)),
                "fold": fold_name,
                "train_rows": fold_train_rows,
                "validation_rows": fold_validation_rows,
                "validation_prior_reconstructed": validation_prior_reconstructed,
                "macro_f1": float(np.mean(class_f1)),
                "minimum_class_f1": float(np.min(class_f1)),
                "non_v_precision": float(precision_score(
                    actual_non_v,
                    predicted_non_v,
                    zero_division=0,
                    sample_weight=fold_metric_weight,
                )),
                "non_v_recall": float(recall_score(
                    actual_non_v,
                    predicted_non_v,
                    zero_division=0,
                    sample_weight=fold_metric_weight,
                )),
                "best_iteration": int(fold_model.get_best_iteration()),
            })
            del fold_model, fold_train_pool, fold_validation_pool

    temporal_feature_fold_scores = pd.DataFrame(feature_fold_rows)
    feature_group_ablation = (
        temporal_feature_fold_scores
        .groupby(["feature_group", "feature_count"], as_index=False)
        .agg(
            worst_fold_macro_f1=("macro_f1", "min"),
            mean_fold_macro_f1=("macro_f1", "mean"),
            worst_fold_minimum_class_f1=("minimum_class_f1", "min"),
            worst_fold_non_v_recall=("non_v_recall", "min"),
            mean_fold_non_v_precision=("non_v_precision", "mean"),
            mean_best_iteration=("best_iteration", "mean"),
        )
        .sort_values(
            [
                "worst_fold_macro_f1",
                "mean_fold_macro_f1",
                "worst_fold_minimum_class_f1",
                "worst_fold_non_v_recall",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )
    display(temporal_feature_fold_scores)
    display(feature_group_ablation)
    SELECTED_FEATURE_GROUP = str(
        feature_group_ablation.iloc[0]["feature_group"]
    )
else:
    temporal_feature_fold_scores = pd.DataFrame()
    feature_group_ablation = pd.DataFrame()
    SELECTED_FEATURE_GROUP = "RULES_NO_PATTERN_CAPPED_DELAY"

MODEL_FEATURE_COLUMNS = list(ablation_groups[SELECTED_FEATURE_GROUP])
MODEL_CATEGORICAL_COLUMNS = [
    c for c in CATEGORICAL_COLUMNS if c in MODEL_FEATURE_COLUMNS
]
print("Selected feature group:", SELECTED_FEATURE_GROUP)
print("Selected model feature count:", len(MODEL_FEATURE_COLUMNS))

pattern_model_features = [
    column for column in MODEL_FEATURE_COLUMNS if column.startswith("pattern_")
]
if pattern_model_features:
    raise RuntimeError(
        "Non-causal pattern features entered the model: "
        + ", ".join(pattern_model_features)
    )
print("Pattern features supplied to model:", pattern_model_features)


# ---------------------------------------------------------
# 3. Train the development model on data available before 2025.
# ---------------------------------------------------------
train_pool = make_pool(
    X_train_model,
    y_train_encoded,
    MODEL_FEATURE_COLUMNS,
    MODEL_CATEGORICAL_COLUMNS,
    weight=train_sample_weight,
)
validation_pool = make_pool(
    X_validation_model,
    y_validation_encoded,
    MODEL_FEATURE_COLUMNS,
    MODEL_CATEGORICAL_COLUMNS,
)
model = CatBoostClassifier(
    **dict(BASE_MODEL_PARAMETERS, iterations=ITERATIONS, verbose=100)
)
model.fit(
    train_pool,
    eval_set=validation_pool,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    use_best_model=True,
)
DEVELOPMENT_BEST_ITERATION = max(1, int(model.get_best_iteration()) + 1)
print("Development best iteration:", DEVELOPMENT_BEST_ITERATION)
print("Model classes:", STATUS_CLASSES)
print("Selected features:", len(MODEL_FEATURE_COLUMNS))


# ---------------------------------------------------------
# 4. Tune fallback-to-V thresholds by worst validation month first.
# ---------------------------------------------------------
def apply_class_thresholds(probabilities, thresholds):
    argmax_indices = probabilities.argmax(axis=1)
    predicted = status_encoder.inverse_transform(argmax_indices)
    selected_probability = probabilities[
        np.arange(len(probabilities)), argmax_indices
    ]
    fallback = np.zeros(len(predicted), dtype=bool)
    for status_class, minimum_probability in thresholds.items():
        if status_class == "V":
            continue
        mask = (
            (predicted == status_class)
            & (selected_probability < float(minimum_probability))
        )
        predicted[mask] = "V"
        fallback |= mask
    return predicted, fallback


validation_probabilities_for_thresholds = model.predict_proba(
    X_validation_model[MODEL_FEATURE_COLUMNS]
)
NON_V_CLASSES = [c for c in STATUS_CLASSES if c != "V"]
CLASS_THRESHOLDS = {c: 0.0 for c in NON_V_CLASSES}
threshold_rows = []

for pass_number in range(1, CLASS_THRESHOLD_PASSES + 1):
    for tuned_class in NON_V_CLASSES:
        candidates = []
        for threshold in CLASS_THRESHOLD_GRID:
            candidate_thresholds = dict(CLASS_THRESHOLDS)
            candidate_thresholds[tuned_class] = float(threshold)
            candidate_predicted, _ = apply_class_thresholds(
                validation_probabilities_for_thresholds,
                candidate_thresholds,
            )
            temporal = temporal_macro_summary(
                y_validation_status,
                candidate_predicted,
                validation_df[SPLIT_DATE_COLUMN],
            )
            overall_macro = f1_score(
                y_validation_status,
                candidate_predicted,
                labels=STATUS_CLASSES,
                average="macro",
                zero_division=0,
            )
            actual_non_v = y_validation_status.ne("V").to_numpy()
            predicted_non_v = candidate_predicted != "V"
            non_v_precision = precision_score(
                actual_non_v, predicted_non_v, zero_division=0
            )
            objective = (
                temporal["worst_monthly_macro_f1"],
                temporal["mean_monthly_macro_f1"],
                float(overall_macro),
                float(non_v_precision),
                -float(threshold),
            )
            candidates.append((objective, float(threshold)))
            threshold_rows.append({
                "pass": pass_number,
                "tuned_class": tuned_class,
                "candidate_threshold": float(threshold),
                "validation_worst_monthly_macro_f1": temporal[
                    "worst_monthly_macro_f1"
                ],
                "validation_mean_monthly_macro_f1": temporal[
                    "mean_monthly_macro_f1"
                ],
                "validation_macro_f1": float(overall_macro),
                "validation_non_v_precision": float(non_v_precision),
            })
        CLASS_THRESHOLDS[tuned_class] = max(candidates)[1]

threshold_tuning_report = pd.DataFrame(threshold_rows)
threshold_tuning_report["selected_final_threshold"] = (
    threshold_tuning_report.apply(
        lambda row: CLASS_THRESHOLDS.get(row["tuned_class"])
        == row["candidate_threshold"],
        axis=1,
    )
)
print("Validation-selected class thresholds:", CLASS_THRESHOLDS)
display(threshold_tuning_report)


Training shape: (372266, 90)
Validation shape: (237745, 90)
Categorical feature count: 18
Numeric feature count: 72
Status in model input: False
power=0.00 worst_month=0.4448 mean_month=0.5791 macro_f1=0.7149
power=0.25 worst_month=0.5505 mean_month=0.6604 macro_f1=0.7718
power=0.50 worst_month=0.4074 mean_month=0.5206 macro_f1=0.6381
power=0.75 worst_month=0.2657 mean_month=0.4587 macro_f1=0.3751
power=1.00 worst_month=0.2659 mean_month=0.4613 macro_f1=0.3739


,prior_correction_power,validation_worst_monthly_macro_f1,validation_mean_monthly_macro_f1,validation_macro_f1,validation_accuracy,best_iteration
0,0.25,0.550523,0.660405,0.771810,0.902236,119
1,0.00,0.444794,0.579081,0.714911,0.895451,139
2,0.50,0.407435,0.520584,0.638134,0.902072,90
3,1.00,0.265916,0.461263,0.373866,0.792008,89
4,0.75,0.265675,0.458731,0.375121,0.790528,100


Selected PRIOR_CORRECTION_POWER: 0.25


,sampled_training_prior,natural_training_prior,training_weight
status,,,
I,0.222427,0.037091,0.639029
MR,0.086981,0.014505,0.639029
N,0.019029,0.003173,0.639029
V,0.671563,0.945231,1.089213


,feature_group,feature_count,fold,train_rows,validation_rows,validation_prior_reconstructed,macro_f1,minimum_class_f1,non_v_precision,non_v_recall,best_iteration
0,RULES_NO_PATTERN_NO_DELAY,79,2024_H1,272212,54448,True,0.624620,0.136152,0.370102,0.323605,286
1,RULES_NO_PATTERN_NO_DELAY,79,2024_H2,326660,45606,True,0.362004,0.085563,0.183994,0.461000,191
2,RULES_NO_PATTERN_NO_DELAY,79,2025_H1,372266,237745,False,0.771810,0.592705,0.919089,0.589996,119
3,RULES_NO_PATTERN_CAPPED_DELAY,79,2024_H1,272212,54448,True,0.624620,0.136152,0.370102,0.323605,286
4,RULES_NO_PATTERN_CAPPED_DELAY,79,2024_H2,326660,45606,True,0.362004,0.085563,0.183994,0.461000,191
5,RULES_NO_PATTERN_CAPPED_DELAY,79,2025_H1,372266,237745,False,0.771810,0.592705,0.919089,0.589996,119
6,RULES_NO_PATTERN_NO_SOURCE,78,2024_H1,272212,54448,True,0.617804,0.134940,0.371440,0.309535,176
7,RULES_NO_PATTERN_NO_SOURCE,78,2024_H2,326660,45606,True,0.363069,0.091395,0.195389,0.483607,164
8,RULES_NO_PATTERN_NO_SOURCE,78,2025_H1,372266,237745,False,0.735561,0.538341,0.920524,0.585161,106


,feature_group,feature_count,worst_fold_macro_f1,mean_fold_macro_f1,worst_fold_minimum_class_f1,worst_fold_non_v_recall,mean_fold_non_v_precision,mean_best_iteration
0,RULES_NO_PATTERN_NO_SOURCE,78,0.363069,0.572145,0.091395,0.309535,0.495784,148.666667
1,RULES_NO_PATTERN_CAPPED_DELAY,79,0.362004,0.586145,0.085563,0.323605,0.491061,198.666667
2,RULES_NO_PATTERN_NO_DELAY,79,0.362004,0.586145,0.085563,0.323605,0.491061,198.666667


Selected feature group: RULES_NO_PATTERN_NO_SOURCE
Selected model feature count: 78
Pattern features supplied to model: []
0:	learn: 1.3270722	test: 1.3261794	best: 1.3261794 (0)	total: 366ms	remaining: 12m 11s
100:	learn: 0.3476033	test: 0.4490209	best: 0.4483850 (99)	total: 28.3s	remaining: 8m 53s
200:	learn: 0.2877713	test: 0.4599481	best: 0.4477664 (106)	total: 58.8s	remaining: 8m 46s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.4477664139
bestIteration = 106

Shrink model to first 107 iterations.
Development best iteration: 107
Model classes: ['I', 'MR', 'N', 'V']
Selected features: 78
Validation-selected class thresholds: {'I': 0.4, 'MR': 0.0, 'N': 0.5}


,pass,tuned_class,candidate_threshold,validation_worst_monthly_macro_f1,validation_mean_monthly_macro_f1,validation_macro_f1,validation_non_v_precision,selected_final_threshold
0,1,I,0.0,0.539707,0.637750,0.735561,0.920524,False
1,1,I,0.3,0.539707,0.637750,0.735561,0.920524,False
2,1,I,0.4,0.539718,0.638285,0.735834,0.920971,True
3,1,I,0.5,0.539097,0.623765,0.735510,0.932001,False
4,1,I,0.6,0.538228,0.623335,0.736081,0.942153,False
5,1,I,0.7,0.447463,0.575911,0.547584,0.569288,False
6,1,I,0.8,0.446862,0.520389,0.542256,0.503028,False
7,1,MR,0.0,0.539718,0.638285,0.735834,0.920971,True
8,1,MR,0.3,0.539718,0.638285,0.735834,0.920971,False
9,1,MR,0.4,0.528196,0.630115,0.727539,0.922630,False


## 9. Evaluate validation with honest baselines

In [12]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)


def evaluate_split(split_name, raw_frame, feature_frame, y_status):
    probabilities = model.predict_proba(feature_frame[MODEL_FEATURE_COLUMNS])
    argmax_indices = probabilities.argmax(axis=1)
    argmax_predicted = status_encoder.inverse_transform(argmax_indices)
    predicted, threshold_fallback = apply_class_thresholds(
        probabilities, CLASS_THRESHOLDS
    )

    seen_mask = y_status.isin(STATUS_CLASSES).to_numpy()
    present_labels = sorted(set(y_status) & set(STATUS_CLASSES))
    encoded_seen = status_encoder.transform(y_status.loc[seen_mask])
    actual_non_v = y_status.ne("V").to_numpy()
    predicted_non_v = predicted != "V"

    metrics = {
        "split": split_name,
        "rows": int(len(y_status)),
        "accuracy": float(accuracy_score(y_status, predicted)),
        "all_v_accuracy_baseline": float(y_status.eq("V").mean()),
        "accuracy_lift_vs_all_v": float(
            accuracy_score(y_status, predicted) - y_status.eq("V").mean()
        ),
        "balanced_accuracy": float(balanced_accuracy_score(y_status, predicted)),
        "macro_f1_present_classes": float(f1_score(
            y_status, predicted, labels=present_labels,
            average="macro", zero_division=0,
        )),
        "macro_f1_all_model_classes": float(f1_score(
            y_status, predicted, labels=STATUS_CLASSES,
            average="macro", zero_division=0,
        )),
        "argmax_macro_f1": float(f1_score(
            y_status, argmax_predicted, labels=STATUS_CLASSES,
            average="macro", zero_division=0,
        )),
        "weighted_f1": float(f1_score(
            y_status, predicted, average="weighted", zero_division=0,
        )),
        "non_v_precision": float(precision_score(
            actual_non_v, predicted_non_v, zero_division=0,
        )),
        "non_v_recall": float(recall_score(
            actual_non_v, predicted_non_v, zero_division=0,
        )),
        "review_rate": float(predicted_non_v.mean()),
        "threshold_fallback_rate": float(threshold_fallback.mean()),
        "false_negative_non_v": int((actual_non_v & ~predicted_non_v).sum()),
        "false_positive_v_as_non_v": int((~actual_non_v & predicted_non_v).sum()),
        "exact_non_v_correct": int(
            (actual_non_v & (y_status.to_numpy() == predicted)).sum()
        ),
        "exact_non_v_accuracy": float(
            (y_status.to_numpy()[actual_non_v] == predicted[actual_non_v]).mean()
        ) if actual_non_v.any() else float("nan"),
        "multiclass_log_loss": float(log_loss(
            encoded_seen,
            probabilities[seen_mask],
            labels=np.arange(len(STATUS_CLASSES)),
        )),
    }

    report = pd.DataFrame(classification_report(
        y_status, predicted, labels=STATUS_CLASSES,
        output_dict=True, zero_division=0,
    )).transpose()
    confusion = pd.DataFrame(
        confusion_matrix(y_status, predicted, labels=STATUS_CLASSES),
        index=pd.Index(STATUS_CLASSES, name="actual_status"),
        columns=pd.Index(STATUS_CLASSES, name="predicted_status"),
    )

    identity_columns = [
        column for column in (
            "id", "created_at", "do2_date", "do2_type", "do2_group",
            "unit_code", "description", "transcription_type",
        )
        if column in raw_frame.columns
    ]
    predictions = raw_frame[identity_columns].copy()
    predictions["created_month"] = parse_datetime_mixed(
        raw_frame[SPLIT_DATE_COLUMN]
    ).dt.to_period("M").astype(str).to_numpy()
    transcription = _normalized_text(raw_frame, "transcription_type")
    predictions["source_segment"] = np.where(
        transcription.eq("S"), "SYSTEM_GENERATED", "DIRECT_OR_OTHER"
    )
    delay_days = (
        parse_datetime_mixed(raw_frame[SPLIT_DATE_COLUMN]).dt.normalize()
        - parse_datetime_mixed(raw_frame["do2_date"]).dt.normalize()
    ).dt.days
    predictions["delay_bucket"] = pd.cut(
        delay_days,
        bins=[-np.inf, -1, 30, 90, 365, np.inf],
        labels=[
            "FUTURE_DO2_DATE", "D00_030", "D031_090",
            "D091_365", "D366_PLUS",
        ],
    ).astype("string").fillna("MISSING").to_numpy()
    for source_column in ("column_a", "column_b", "column_c"):
        predictions[f"has_{source_column}"] = _column_present(
            raw_frame, source_column
        ).to_numpy()

    pattern_keys = feature_frame[PATTERN_KEY_COLUMN].astype(str)
    predictions["bill_pattern_key"] = pattern_keys.to_numpy()
    predictions["pattern_seen_in_training"] = pattern_keys.isin(
        TRAIN_PATTERN_KEYS
    ).to_numpy()
    predictions["actual_status"] = y_status.to_numpy()
    predictions["argmax_predicted_status"] = argmax_predicted
    predictions["predicted_status"] = predicted
    selected_indices = status_encoder.transform(predicted)
    predictions["predicted_status_probability"] = probabilities[
        np.arange(len(probabilities)), selected_indices
    ]
    predictions["threshold_fallback_to_v"] = threshold_fallback
    predictions["is_correct"] = predictions["actual_status"].eq(
        predictions["predicted_status"]
    )
    for class_index, status_class in enumerate(STATUS_CLASSES):
        predictions[f"probability_status_{status_class}"] = probabilities[:, class_index]

    temporal = temporal_macro_summary(
        y_status, predicted, raw_frame[SPLIT_DATE_COLUMN]
    )
    metrics["equal_weight_monthly_macro_f1"] = temporal[
        "mean_monthly_macro_f1"
    ]
    metrics["worst_monthly_macro_f1"] = temporal[
        "worst_monthly_macro_f1"
    ]
    metrics["seen_pattern_percentage"] = float(
        predictions["pattern_seen_in_training"].mean() * 100
    )
    return metrics, report, confusion, predictions


def build_segment_metrics(predictions):
    rows = []
    dimensions = (
        "created_month", "source_segment", "delay_bucket",
        "pattern_seen_in_training",
    )
    for dimension in dimensions:
        for value, group in predictions.groupby(dimension, dropna=False):
            actual = group["actual_status"]
            predicted = group["predicted_status"]
            rows.append({
                "dimension": dimension,
                "segment": str(value),
                "rows": int(len(group)),
                "accuracy": float(accuracy_score(actual, predicted)),
                "macro_f1": float(f1_score(
                    actual, predicted, labels=STATUS_CLASSES,
                    average="macro", zero_division=0,
                )),
                "review_rate": float(predicted.ne("V").mean()),
            })
    return pd.DataFrame(rows)


def build_segment_class_metrics(predictions):
    rows = []
    dimensions = ("source_segment", "delay_bucket", "pattern_seen_in_training")
    for dimension in dimensions:
        for segment, group in predictions.groupby(dimension, dropna=False):
            precision, recall, f1, support = precision_recall_fscore_support(
                group["actual_status"], group["predicted_status"],
                labels=STATUS_CLASSES, zero_division=0,
            )
            for index, status_class in enumerate(STATUS_CLASSES):
                rows.append({
                    "dimension": dimension,
                    "segment": str(segment),
                    "status": status_class,
                    "precision": float(precision[index]),
                    "recall": float(recall[index]),
                    "f1": float(f1[index]),
                    "support": int(support[index]),
                })
    return pd.DataFrame(rows)


def build_error_pattern_report(predictions, top_n=5000):
    errors = predictions.loc[~predictions["is_correct"]].copy()
    grouping = [
        "actual_status", "predicted_status", "description",
        "do2_group", "do2_type", "source_segment", "delay_bucket",
        "has_column_a", "has_column_b", "has_column_c",
        "pattern_seen_in_training",
    ]
    grouping = [column for column in grouping if column in errors]
    return (
        errors.groupby(grouping, dropna=False)
        .size()
        .reset_index(name="error_rows")
        .sort_values("error_rows", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


(
    validation_metrics,
    validation_report,
    validation_confusion,
    validation_predictions,
) = evaluate_split(
    "validation", validation_df, X_validation_model, y_validation_status
)
validation_segment_metrics = build_segment_metrics(validation_predictions)
validation_segment_class_metrics = build_segment_class_metrics(validation_predictions)
validation_error_patterns = build_error_pattern_report(validation_predictions)

display(pd.DataFrame([validation_metrics]))
display(validation_report)
display(validation_confusion)
display(validation_predictions.head())
display(validation_segment_metrics)
display(validation_segment_class_metrics)
display(validation_error_patterns.head(50))


,split,rows,accuracy,all_v_accuracy_baseline,accuracy_lift_vs_all_v,balanced_accuracy,macro_f1_present_classes,macro_f1_all_model_classes,argmax_macro_f1,weighted_f1,...,review_rate,threshold_fallback_rate,false_negative_non_v,false_positive_v_as_non_v,exact_non_v_correct,exact_non_v_accuracy,multiclass_log_loss,equal_weight_monthly_macro_f1,worst_monthly_macro_f1,seen_pattern_percentage
0,validation,237745,0.900827,0.789447,0.11138,0.651112,0.6965,0.6965,0.735561,0.892143,...,0.132592,0.001253,21016,2481,28961,0.578549,0.447766,0.644276,0.587338,99.298408


,precision,recall,f1-score,support
I,0.939387,0.579754,0.717001,48305.000000
MR,0.479777,0.613191,0.538341,1122.000000
N,0.967509,0.424723,0.590308,631.000000
V,0.898090,0.986781,0.940349,187687.000000
accuracy,0.900827,0.900827,0.900827,0.900827
macro avg,0.821191,0.651112,0.696500,237745.000000
weighted avg,0.904691,0.900827,0.892143,237745.000000


predicted_status,I,MR,N,V
actual_status,,,,
I,28005,39,0,20261
MR,13,688,2,419
N,26,1,268,336
V,1768,706,7,185206


,id,created_at,do2_date,do2_type,do2_group,unit_code,description,transcription_type,created_month,source_segment,...,actual_status,argmax_predicted_status,predicted_status,predicted_status_probability,threshold_fallback_to_v,is_correct,probability_status_I,probability_status_MR,probability_status_N,probability_status_V
0,14286673,2025-01-15 17:36:31.769,2018-09-25,0,IA,64B014,ADPOST,S,2025-01,SYSTEM_GENERATED,...,I,I,I,0.676123,False,True,0.676123,0.064352,0.038802,0.220723
1,15373818,2025-01-06 18:08:31.402,2024-12-25,0,IA,92B005,HAUCA,D,2025-01,DIRECT_OR_OTHER,...,V,V,V,0.833308,False,True,0.102298,0.041420,0.022974,0.833308
2,15369314,2025-01-09 11:15:15.561,2024-12-16,0,IA,64B015,TPTL,D,2025-01,DIRECT_OR_OTHER,...,V,V,V,0.918458,False,True,0.037939,0.024430,0.019173,0.918458
3,15485819,2025-03-05 12:18:05.809,2025-01-08,0,IA,HWS166,CANCEL,D,2025-03,DIRECT_OR_OTHER,...,V,V,V,0.832429,False,True,0.054545,0.064144,0.048882,0.832429
4,15477770,2025-03-10 11:16:28.717,2025-02-15,0,IA,92B061,CANCEL,D,2025-03,DIRECT_OR_OTHER,...,V,V,V,0.850400,False,True,0.051509,0.052417,0.045673,0.850400


,dimension,segment,rows,accuracy,macro_f1,review_rate
0,created_month,2025-01,64665,0.898740,0.653436,0.432846
1,created_month,2025-02,29543,0.970044,0.660655,0.031547
2,created_month,2025-03,24667,0.968298,0.603238,0.026270
3,created_month,2025-04,42591,0.983541,0.681744,0.019769
4,created_month,2025-05,23995,0.976829,0.679245,0.023630
5,created_month,2025-06,52284,0.730204,0.587338,0.010405
6,source_segment,DIRECT_OR_OTHER,153294,0.978492,0.715786,0.023517
7,source_segment,SYSTEM_GENERATED,84451,0.759849,0.378820,0.330582
8,delay_bucket,D00_030,179071,0.881706,0.596614,0.012783
9,delay_bucket,D031_090,23327,0.977065,0.751043,0.027222


,dimension,segment,status,precision,recall,f1,support
0,source_segment,DIRECT_OR_OTHER,I,0.529039,0.459001,0.491538,2183
1,source_segment,DIRECT_OR_OTHER,MR,0.479777,0.613191,0.538341,1122
2,source_segment,DIRECT_OR_OTHER,N,0.967509,0.748603,0.844094,358
3,source_segment,DIRECT_OR_OTHER,V,0.988977,0.989360,0.989169,149631
4,source_segment,SYSTEM_GENERATED,I,0.967225,0.585469,0.729417,46122
5,source_segment,SYSTEM_GENERATED,MR,0.000000,0.000000,0.000000,0
6,source_segment,SYSTEM_GENERATED,N,0.000000,0.000000,0.000000,273
7,source_segment,SYSTEM_GENERATED,V,0.657439,0.976640,0.785863,38056
8,delay_bucket,D00_030,I,0.532587,0.025761,0.049145,20302
9,delay_bucket,D00_030,MR,0.493043,0.637795,0.556155,889


,actual_status,predicted_status,description,do2_group,do2_type,source_segment,delay_bucket,has_column_a,has_column_b,has_column_c,pattern_seen_in_training,error_rows
0,I,V,LRA,IA,0,SYSTEM_GENERATED,D00_030,0,0,0,True,13545
1,I,V,INCR,IA,0,SYSTEM_GENERATED,D00_030,0,0,1,True,5503
2,V,I,INCR,IA,0,SYSTEM_GENERATED,D366_PLUS,0,0,1,True,387
3,V,MR,CHFMNM,IIA,0,DIRECT_OR_OTHER,D00_030,1,1,1,True,220
4,V,I,EXTWK,IA,0,SYSTEM_GENERATED,D00_030,1,0,0,True,205
5,V,MR,PAN,IIA,0,DIRECT_OR_OTHER,D00_030,1,1,1,True,158
6,V,I,CEA,IA,0,DIRECT_OR_OTHER,D366_PLUS,1,1,1,True,151
7,N,V,HRA,IA,0,SYSTEM_GENERATED,D366_PLUS,1,0,0,True,122
8,N,V,CILQ,IA,0,SYSTEM_GENERATED,D366_PLUS,1,0,1,True,111
9,MR,V,ACCLVE,IIB,0,DIRECT_OR_OTHER,D00_030,1,1,1,True,110


## 8A. Tune the safety decision policy

The multiclass model still estimates `I`, `MR`, `N`, and `V`. The operational
decision layer adds `REVIEW`. Only a sufficiently confident `V` is
auto-approved. An uncertain non-V prediction is **never converted to V**.

The V threshold is selected using validation only. The untouched test period
is used once for the final safety assessment.

In [13]:
SAFETY_OUTPUT_CLASSES = ["I", "MR", "N", "V", "REVIEW"]


def probability_matrix_from_predictions(predictions):
    return predictions[
        [f"probability_status_{status}" for status in STATUS_CLASSES]
    ].to_numpy(dtype=float)


def apply_safety_policy(probabilities, v_threshold, class_thresholds):
    """Return I/MR/N/V/REVIEW without ever using V as an uncertainty fallback."""
    probabilities = np.asarray(probabilities, dtype=float)
    argmax_index = probabilities.argmax(axis=1)
    argmax_status = status_encoder.inverse_transform(argmax_index)
    argmax_probability = probabilities[np.arange(len(probabilities)), argmax_index]
    v_index = STATUS_CLASSES.index("V")
    v_probability = probabilities[:, v_index]

    decisions = np.full(len(probabilities), "REVIEW", dtype=object)
    for non_v_status in ("I", "MR", "N"):
        class_cutoff = float(class_thresholds.get(non_v_status, 0.0))
        confident_class = (
            (argmax_status == non_v_status)
            & (argmax_probability >= class_cutoff)
        )
        decisions[confident_class] = non_v_status

    unresolved = decisions == "REVIEW"
    decisions[unresolved & (v_probability >= float(v_threshold))] = "V"
    return decisions, v_probability, argmax_status, argmax_probability


def safety_metrics(y_true, decisions):
    actual = np.asarray(y_true, dtype=str)
    decisions = np.asarray(decisions, dtype=str)
    actual_non_v = actual != "V"
    predicted_v = decisions == "V"
    predicted_review = decisions == "REVIEW"
    predicted_named_non_v = np.isin(decisions, ["I", "MR", "N"])
    missed_non_v = actual_non_v & predicted_v
    auto_valid = predicted_v
    exact_non_v = actual_non_v & (actual == decisions)
    return {
        "rows": int(len(actual)),
        "auto_valid_rate": float(auto_valid.mean()),
        "review_rate": float(predicted_review.mean()),
        "named_non_v_rate": float(predicted_named_non_v.mean()),
        "non_v_safety_recall": float(1.0 - missed_non_v.mean() / actual_non_v.mean())
            if actual_non_v.any() else float("nan"),
        "false_negative_non_v_as_v": int(missed_non_v.sum()),
        "false_positive_v_not_auto_valid": int(((actual == "V") & ~predicted_v).sum()),
        "v_auto_approval_precision": float((actual[auto_valid] == "V").mean())
            if auto_valid.any() else float("nan"),
        "v_auto_approval_recall": float((decisions[actual == "V"] == "V").mean())
            if (actual == "V").any() else float("nan"),
        "exact_non_v_accuracy": float(exact_non_v.sum() / actual_non_v.sum())
            if actual_non_v.any() else float("nan"),
    }


def tune_v_safety_threshold(
    y_true,
    probabilities,
    *,
    max_non_v_to_v_rate=MAX_NON_V_TO_V_RATE,
    minimum_auto_v_precision=MIN_AUTO_V_PRECISION,
):
    """Learn a per-bill V threshold without assuming future class percentages."""
    actual = np.asarray(y_true, dtype=str)
    probabilities = np.asarray(probabilities, dtype=float)
    if "V" not in STATUS_CLASSES:
        raise ValueError("The trained model does not contain status V")

    v_index = STATUS_CLASSES.index("V")
    v_probabilities = probabilities[:, v_index]
    actual_non_v = actual != "V"
    actual_v = actual == "V"
    if not actual_non_v.any() or not actual_v.any():
        raise ValueError("Validation must contain both V and non-V rows")

    # Sort once and calculate all exact probability boundaries cumulatively.
    # This is O(n log n), rather than applying every threshold to every row.
    order = np.argsort(-v_probabilities, kind="mergesort")
    sorted_probability = v_probabilities[order]
    sorted_actual_v = actual_v[order].astype(np.int64)
    sorted_actual_non_v = actual_non_v[order].astype(np.int64)
    cumulative_v = np.cumsum(sorted_actual_v)
    cumulative_non_v = np.cumsum(sorted_actual_non_v)

    # For ties, evaluate only after the final row with that probability because
    # `probability_V >= threshold` includes the complete tied group.
    group_end = np.r_[
        sorted_probability[:-1] != sorted_probability[1:],
        True,
    ]
    boundary_index = np.flatnonzero(group_end)
    auto_v_rows = boundary_index + 1
    auto_v_actual_v = cumulative_v[boundary_index]
    auto_v_actual_non_v = cumulative_non_v[boundary_index]

    report = pd.DataFrame({
        "v_threshold": sorted_probability[boundary_index],
        "validation_rows": int(len(actual)),
        "actual_non_v_rows": int(actual_non_v.sum()),
        "actual_v_rows": int(actual_v.sum()),
        "non_v_declared_v": auto_v_actual_non_v,
        "auto_v_rows": auto_v_rows,
    })
    report["non_v_to_v_rate"] = (
        report["non_v_declared_v"] / int(actual_non_v.sum())
    )
    report["non_v_safety_recall"] = 1.0 - report["non_v_to_v_rate"]
    report["auto_v_precision"] = auto_v_actual_v / auto_v_rows
    report["v_auto_approval_recall"] = (
        auto_v_actual_v / int(actual_v.sum())
    )
    eligible = report.loc[
        report["non_v_to_v_rate"].le(max_non_v_to_v_rate)
        & report["auto_v_precision"].ge(minimum_auto_v_precision)
    ].copy()
    if eligible.empty:
        print("WARNING: no threshold met both safety conditions.")
        print("Automatic V approval is disabled; unresolved rows become REVIEW.")
        return float(np.nextafter(1.0, 2.0)), report

    selected = eligible.sort_values(
        ["v_auto_approval_recall", "v_threshold"],
        ascending=[False, False],
    ).iloc[0]
    return float(selected["v_threshold"]), report


validation_probability_matrix = probability_matrix_from_predictions(
    validation_predictions
)
SAFETY_V_THRESHOLD, safety_threshold_tuning = tune_v_safety_threshold(
    y_validation_status, validation_probability_matrix
)
validation_safety_decisions, _, _, _ = apply_safety_policy(
    validation_probability_matrix, SAFETY_V_THRESHOLD, CLASS_THRESHOLDS
)
validation_safety_metrics = safety_metrics(
    y_validation_status, validation_safety_decisions
)
validation_safety_confusion = pd.DataFrame(
    confusion_matrix(
        y_validation_status,
        validation_safety_decisions,
        labels=SAFETY_OUTPUT_CLASSES,
    ),
    index=pd.Index(SAFETY_OUTPUT_CLASSES, name="actual_status"),
    columns=pd.Index(SAFETY_OUTPUT_CLASSES, name="decision"),
)
validation_predictions["safety_decision"] = validation_safety_decisions

print("Selected per-bill safety V threshold:", SAFETY_V_THRESHOLD)
display(pd.DataFrame([validation_safety_metrics]))
display(validation_safety_confusion)
display(safety_threshold_tuning)


Automatic V approval is disabled; unresolved rows become REVIEW.
Selected per-bill safety V threshold: 1.0000000000000002


,rows,auto_valid_rate,review_rate,named_non_v_rate,non_v_safety_recall,false_negative_non_v_as_v,false_positive_v_not_auto_valid,v_auto_approval_precision,v_auto_approval_recall,exact_non_v_accuracy
0,237745,0.0,0.867408,0.132592,1.0,0,187687,NaN,0.0,0.578549


decision,I,MR,N,V,REVIEW
actual_status,,,,,
I,28005,39,0,0,20261
MR,13,688,2,0,419
N,26,1,268,0,336
V,1768,706,7,0,185206
REVIEW,0,0,0,0,0


,v_threshold,validation_rows,actual_non_v_rows,actual_v_rows,non_v_declared_v,auto_v_rows,non_v_to_v_rate,non_v_safety_recall,auto_v_precision,v_auto_approval_recall
0,0.953595,237745,50058,187687,9908,29680,0.197930,0.802070,0.666173,0.105346
1,0.953355,237745,50058,187687,9929,29748,0.198350,0.801650,0.666230,0.105596
2,0.953344,237745,50058,187687,10594,31685,0.211635,0.788365,0.665646,0.112373
3,0.953104,237745,50058,187687,12221,36523,0.244137,0.755863,0.665389,0.129482
4,0.951955,237745,50058,187687,12269,36680,0.245096,0.754904,0.665513,0.130062
...,...,...,...,...,...,...,...,...,...,...
15477,0.082052,237745,50058,187687,49999,237677,0.998821,0.001179,0.789635,0.999952
15478,0.081881,237745,50058,187687,50000,237678,0.998841,0.001159,0.789631,0.999952
15479,0.081704,237745,50058,187687,50057,237735,0.999980,0.000020,0.789442,0.999952
15480,0.081404,237745,50058,187687,50057,237737,0.999980,0.000020,0.789444,0.999963


In [14]:
# Preserve development results, then refit the deployment model through June 2025.
TRAINING_ROWS = int(len(train_df))
VALIDATION_ROWS = int(len(validation_df))
TRAINING_CLASS_COUNTS = {
    str(key): int(value)
    for key, value in y_train_status.value_counts().sort_index().to_dict().items()
}

if REFIT_WITH_VALIDATION:
    refit_X = pd.concat(
        [
            X_train_model[MODEL_FEATURE_COLUMNS],
            X_validation_model[MODEL_FEATURE_COLUMNS],
        ],
        ignore_index=True,
    )
    refit_y_status = pd.concat(
        [
            y_train_status.reset_index(drop=True),
            y_validation_status.reset_index(drop=True),
        ],
        ignore_index=True,
    )

    # Training rows keep their automatically selected V-sampling correction.
    # The natural-distribution validation rows receive weight 1.0.
    refit_weight = np.concatenate([
        train_sample_weight,
        np.ones(len(y_validation_status), dtype=float),
    ])
    refit_weight = refit_weight / refit_weight.mean()
    refit_pool = Pool(
        data=refit_X,
        label=status_encoder.transform(refit_y_status),
        cat_features=MODEL_CATEGORICAL_COLUMNS,
        weight=refit_weight,
    )
    deployment_model = CatBoostClassifier(
        **dict(
            BASE_MODEL_PARAMETERS,
            iterations=DEVELOPMENT_BEST_ITERATION,
            verbose=100,
        )
    )
    deployment_model.fit(refit_pool)
    model = deployment_model
    DEPLOYMENT_TRAINING_ROWS = int(len(refit_y_status))
    DEPLOYMENT_TRAINING_END = VALIDATION_END
    print(
        "Deployment model refit through:",
        DEPLOYMENT_TRAINING_END,
        "rows:",
        DEPLOYMENT_TRAINING_ROWS,
    )
    del refit_X, refit_y_status, refit_weight, refit_pool
else:
    DEPLOYMENT_TRAINING_ROWS = TRAINING_ROWS
    DEPLOYMENT_TRAINING_END = TRAIN_END

del train_features, validation_features, train_df, validation_df
del X_train_model, X_validation_model, train_pool, validation_pool
del all_validation_pool, development_X, development_y_status, development_timestamps
_ = gc.collect()


0:	learn: 1.3240361	total: 274ms	remaining: 29s
100:	learn: 0.3160954	total: 24.9s	remaining: 1.48s
106:	learn: 0.3077997	total: 26.3s	remaining: 0us
Deployment model refit through: 2025-07-01 rows: 610011


## 10. Evaluate the untouched 2025 test split

Do not change features, weights, thresholds, or model parameters after reading
this section. A change requires a new final holdout period.

In [15]:
test_raw = load_or_cache_enriched(
    "test",
    TEST_START,
    TEST_END,
    MAX_TEST_ROWS,
    lambda: load_holdout_data(TEST_START, TEST_END, MAX_TEST_ROWS),
)
test_df, test_cleaning_report = clean_split(
    test_raw, "test", drop_incomplete_v_labels=False
)
del test_raw
_ = gc.collect()

y_test_status = normalize_status(test_df["status"])
X_test_raw = test_df.drop(
    columns=["status", "target", "is_non_valid"], errors="ignore"
).copy()
test_features = build_features(
    X_test_raw,
    expected_columns=FEATURE_COLUMNS,
    expected_categorical=CATEGORICAL_COLUMNS,
    expected_text=TEXT_COLUMNS,
)

unseen_test_classes = sorted(set(y_test_status) - set(STATUS_CLASSES))
if unseen_test_classes:
    raise ValueError(
        "Test contains statuses the model cannot predict: "
        + ", ".join(unseen_test_classes)
    )

test_split_check = validate_split(test_df, "test", TEST_START, TEST_END)
TEST_IDS = frozenset(test_df["id"].tolist())
train_test_overlap = len(TRAIN_IDS & TEST_IDS)
validation_test_overlap = len(VALIDATION_IDS & TEST_IDS)
if train_test_overlap or validation_test_overlap:
    raise ValueError(
        "Split id overlap detected: "
        f"train/test={train_test_overlap:,}, "
        f"validation/test={validation_test_overlap:,}"
    )

X_test_model = prepare_model_frame(test_features.frame)
test_metrics, test_report, test_confusion, test_predictions = evaluate_split(
    "test", test_df, X_test_model, y_test_status
)
test_segment_metrics = build_segment_metrics(test_predictions)
test_segment_class_metrics = build_segment_class_metrics(test_predictions)
test_error_patterns = build_error_pattern_report(test_predictions)

display(pd.DataFrame([test_cleaning_report]))
display(pd.DataFrame([test_split_check]))
print("Train/test id overlap:", train_test_overlap)
print("Validation/test id overlap:", validation_test_overlap)
display(split_profile(test_df, "test"))
display(pd.DataFrame([test_metrics]))
display(test_report)
display(test_confusion)
display(test_predictions.head())
display(test_segment_metrics)
display(test_segment_class_metrics)
display(test_error_patterns.head(50))


# Safety evaluation uses the untouched test labels only for reporting; no
# threshold, model, or feature is changed after seeing these results.
test_probability_matrix = probability_matrix_from_predictions(test_predictions)
test_safety_decisions, _, _, _ = apply_safety_policy(
    test_probability_matrix, SAFETY_V_THRESHOLD, CLASS_THRESHOLDS
)
test_safety_metrics = safety_metrics(y_test_status, test_safety_decisions)
test_safety_confusion = pd.DataFrame(
    confusion_matrix(
        y_test_status,
        test_safety_decisions,
        labels=SAFETY_OUTPUT_CLASSES,
    ),
    index=pd.Index(SAFETY_OUTPUT_CLASSES, name="actual_status"),
    columns=pd.Index(SAFETY_OUTPUT_CLASSES, name="decision"),
)
test_predictions["safety_decision"] = test_safety_decisions

print("Safety decision results (the decisive production-readiness check):")
display(pd.DataFrame([test_safety_metrics]))
display(test_safety_confusion)


Loading cached test data: /home/ruthikreddy/Desktop/do2/do2_status_prediction/data/model_cache/time_safe_v7_java_n_features_no_pattern_model_test_2025-07-01_2026-01-01_250000.parquet
Loaded cached test rows in 0.2 seconds


,split,initial_rows,unparseable_created_at_rows,missing_created_at_rows,sentinel_created_at_rows,malformed_created_at_rows,excluded_status_rows,excluded_lifecycle_status_rows,excluded_do2_group_rows,incomplete_v_rows_found,incomplete_v_rows_removed,final_rows
0,test,250000,0,0,0,0,13307,13307,0,0,0,236693


,split,rows,unique_ids,minimum_created_at,maximum_created_at
0,test,236693,236693,2025-07-01 10:27:57.447,2025-12-30 01:52:25.482837


Train/test id overlap: 0
Validation/test id overlap: 0


,split,status,rows,percentage
0,test,V,231664,97.875307
1,test,I,3951,1.669251
2,test,N,559,0.236171
3,test,MR,519,0.219271


...

,dimension,segment,status,precision,recall,f1,support
0,source_segment,DIRECT_OR_OTHER,I,0.202353,0.046701,0.075888,3683
1,source_segment,DIRECT_OR_OTHER,MR,0.760870,0.202312,0.319635,519
2,source_segment,DIRECT_OR_OTHER,N,0.986595,0.658318,0.789700,559
3,source_segment,DIRECT_OR_OTHER,V,0.978594,0.996310,0.987372,187807
4,source_segment,SYSTEM_GENERATED,I,0.003926,0.111940,0.007585,268
5,source_segment,SYSTEM_GENERATED,MR,0.000000,0.000000,0.000000,0
6,source_segment,SYSTEM_GENERATED,N,0.000000,0.000000,0.000000,0
7,source_segment,SYSTEM_GENERATED,V,0.993476,0.826436,0.902290,43857
8,delay_bucket,D00_030,I,0.005222,0.020578,0.008330,1798
9,delay_bucket,D00_030,MR,0.883495,0.209195,0.338290,435


,actual_status,predicted_status,description,do2_group,do2_type,source_segment,delay_bucket,has_column_a,has_column_b,has_column_c,pattern_seen_in_training,error_rows
0,V,I,INCR,IA,0,SYSTEM_GENERATED,D00_030,0,0,1,True,6680
1,V,I,INCR,IA,0,SYSTEM_GENERATED,D366_PLUS,0,0,1,True,693
2,V,I,APPT,IA,0,DIRECT_OR_OTHER,D00_030,1,0,0,True,338
3,I,V,CANCEL,IA,0,DIRECT_OR_OTHER,D00_030,1,0,0,True,257
4,I,V,ACCLVE,IIB,0,DIRECT_OR_OTHER,D00_030,1,1,1,True,248
5,V,I,INCR,IA,0,SYSTEM_GENERATED,D091_365,0,0,1,True,148
6,I,V,CANCEL,IA,0,DIRECT_OR_OTHER,D031_090,1,0,0,True,143
7,I,V,CANCEL,IA,0,DIRECT_OR_OTHER,D091_365,1,0,0,True,119
8,I,V,INCR,IA,0,SYSTEM_GENERATED,D00_030,0,0,1,True,112
9,MR,V,RDAILY,IA,0,DIRECT_OR_OTHER,D00_030,1,1,1,True,98


Safety decision results (the decisive production-readiness check):


,rows,auto_valid_rate,review_rate,named_non_v_rate,non_v_safety_recall,false_negative_non_v_as_v,false_positive_v_not_auto_valid,v_auto_approval_precision,v_auto_approval_recall,exact_non_v_accuracy
0,236693,0.0,0.961963,0.038037,1.0,0,231664,NaN,0.0,0.134222


decision,I,MR,N,V,REVIEW
actual_status,,,,,
I,202,18,0,0,3731
MR,0,105,5,0,409
N,0,0,368,0,191
V,8290,15,0,0,223359
REVIEW,0,0,0,0,0


In [16]:
def show_status_distribution(status_series, split_name):
    result = (
        status_series
        .fillna("MISSING")
        .astype(str)
        .str.strip()
        .str.upper()
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="rows")
    )

    result["percentage"] = (
        result["rows"] / result["rows"].sum() * 100
    )

    result.insert(0, "split", split_name)

    return result


status_distribution = pd.concat(
    [
        show_status_distribution(y_train_status, "train"),
        show_status_distribution(y_validation_status, "validation"),
        show_status_distribution(y_test_status, "test"),
    ],
    ignore_index=True,
)

display(status_distribution)


,split,status,rows,percentage
0,train,V,250000,67.156281
1,train,I,82802,22.242697
2,train,MR,32380,8.698081
3,train,N,7084,1.902940
4,validation,V,187687,78.944668
5,validation,I,48305,20.317988
6,validation,MR,1122,0.471934
7,validation,N,631,0.265410
8,test,V,231664,97.875307
9,test,I,3951,1.669251


## 11. Deployment-model feature importance

In [17]:
feature_importance = (
    pd.DataFrame({
        "feature": MODEL_FEATURE_COLUMNS,
        "importance": model.get_feature_importance(),
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print(feature_importance)


                                     feature  importance
0                                description   21.220338
1               submission_delay_days_capped   18.419089
2                   from_date_after_do2_date    8.181389
3                               has_column_c    4.510970
4                               has_column_a    3.767285
..                                       ...         ...
73  i_pay_fixation_candidate_support_missing    0.000000
74            i_less_than_six_months_service    0.000000
75           i_less_than_eight_years_service    0.000000
76         v_verified_static_violation_count    0.000000
77       v_all_verified_static_checks_passed    0.000000

[78 rows x 2 columns]


## 12. Save the model and reproducibility reports

In [18]:
if not SAVE_ARTIFACTS:
    raise RuntimeError("Set SAVE_ARTIFACTS=True to save the final model")

MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "do2_catboost_multiclass.cbm"
metadata_path = MODEL_DIR / "model_metadata_multiclass.json"
model.save_model(model_path)

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_version": datetime.now(timezone.utc).strftime(
        "do2-catboost-multiclass-%Y%m%dT%H%M%SZ"
    ),
    "model_source": "do2_file_5_complete_safety_review.ipynb",
    "feature_set_version": FEATURE_SET_VERSION,
    "model_type": "multiclass_status",
    "model_target": f"normalized status in {VALID_STATUSES}",
    "target_mode": TARGET_MODE,
    "classes": STATUS_CLASSES,
    "engineered_feature_columns": FEATURE_COLUMNS,
    "feature_columns": MODEL_FEATURE_COLUMNS,
    "selected_feature_group": SELECTED_FEATURE_GROUP,
    "categorical_columns": MODEL_CATEGORICAL_COLUMNS,
    "text_columns": TEXT_COLUMNS,
    "required_input_columns": REQUIRED_INPUT_COLUMNS,
    "candidate_columns": CANDIDATE_COLUMNS,
    "features_dropped": sorted(FEATURES_TO_DROP),
    "uninformative_training_features_dropped": UNINFORMATIVE_TRAIN_FEATURES,
    "rule_feature_columns": [c for c in FEATURE_COLUMNS if c.startswith(("n_", "i_", "mr_", "v_"))],
    "rule_source_tables": ["do2", "employee", "status_code", "lpc_register", "appendix_j", "family_detail"],
    "data_sources": {
        "train": f"{settings.db_schema}.{TRAIN_TABLE}",
        "validation": f"{settings.db_schema}.{SOURCE_TABLE}",
        "test": f"{settings.db_schema}.{SOURCE_TABLE}",
    },
    "split_date_column": SPLIT_DATE_COLUMN,
    "training_window": [TRAIN_START, TRAIN_END],
    "validation_window": [VALIDATION_START, VALIDATION_END],
    "test_window": [TEST_START, TEST_END],
    "prior_correction_power_grid": list(PRIOR_CORRECTION_POWER_GRID),
    "prior_correction_power_selected": BEST_PRIOR_CORRECTION_POWER,
    "prior_selection_metric": "worst_monthly_macro_f1_then_mean_monthly_macro_f1",
    "feature_selection_metric": "worst_temporal_fold_macro_f1_then_mean_temporal_fold_macro_f1",
    "stability_folds": [list(fold) for fold in STABILITY_FOLDS],
    "monthly_metrics_policy": "diagnostic only; never a model feature",
    "class_thresholds": CLASS_THRESHOLDS,
    "safety_policy": {
        "review_class": "REVIEW",
        "v_auto_approval_threshold": SAFETY_V_THRESHOLD,
        "max_non_v_to_v_rate": MAX_NON_V_TO_V_RATE,
        "minimum_auto_v_precision": MIN_AUTO_V_PRECISION,
        "uncertainty_fallback": "REVIEW",
    },
    "pattern_key_column": PATTERN_KEY_COLUMN,
    "forbidden_calendar_features": sorted(FORBIDDEN_CALENDAR_FEATURES),
    "created_at_caveat": "Java may reset created_at during pending reprocessing; lower id plus earlier/equal created_at is the conservative history proxy",
    "history_status_policy": "No prior do2.status or processed_date used",
    "appendix_j_historical_features_enabled": USE_CURRENT_APPENDIX_J_FOR_HISTORICAL_ROWS,
    "max_train_v_rows": MAX_TRAIN_V_ROWS,
    "max_validation_rows": MAX_VALIDATION_ROWS,
    "max_test_rows": MAX_TEST_ROWS,
    "training_rows": TRAINING_ROWS,
    "validation_rows": VALIDATION_ROWS,
    "deployment_training_rows": DEPLOYMENT_TRAINING_ROWS,
    "deployment_training_end": DEPLOYMENT_TRAINING_END,
    "refit_with_validation": REFIT_WITH_VALIDATION,
    "test_rows": int(len(test_df)),
    "split_overlap_counts": {
        "train_validation": train_validation_overlap,
        "train_test": train_test_overlap,
        "validation_test": validation_test_overlap,
    },
    "training_class_counts": TRAINING_CLASS_COUNTS,
    "unseen_validation_classes": unseen_validation_classes,
    "unseen_test_classes": unseen_test_classes,
    "development_best_iteration": DEVELOPMENT_BEST_ITERATION,
    "deployment_iterations": int(model.tree_count_),
    "model_parameters": model.get_params(),
    "dependency_versions": {
        package: importlib.metadata.version(package)
        for package in (
            "catboost",
            "pandas",
            "numpy",
            "scikit-learn",
            "pyarrow",
        )
    },
    "metrics": {
        "validation": validation_metrics,
        "test": test_metrics,
        "validation_safety": validation_safety_metrics,
        "test_safety": test_safety_metrics,
    },
}

with metadata_path.open("w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=2)

validation_report.to_csv(MODEL_DIR / "validation_classification_report.csv")
validation_confusion.to_csv(MODEL_DIR / "validation_confusion_matrix.csv")
validation_predictions.to_parquet(
    MODEL_DIR / "validation_predictions_multiclass.parquet",
    index=False,
)

test_report.to_csv(MODEL_DIR / "test_classification_report.csv")
test_confusion.to_csv(MODEL_DIR / "test_confusion_matrix.csv")
test_predictions.to_parquet(
    MODEL_DIR / "test_predictions_multiclass.parquet",
    index=False,
)
safety_threshold_tuning.to_csv(
    MODEL_DIR / "safety_v_threshold_tuning.csv", index=False
)
validation_safety_confusion.to_csv(
    MODEL_DIR / "validation_safety_confusion_matrix.csv"
)
test_safety_confusion.to_csv(MODEL_DIR / "test_safety_confusion_matrix.csv")
feature_importance.to_csv(MODEL_DIR / "feature_importance.csv", index=False)
pattern_purity_report.to_csv(MODEL_DIR / "training_pattern_purity.csv", index=False)
threshold_tuning_report.to_csv(MODEL_DIR / "class_threshold_tuning.csv", index=False)
validation_segment_class_metrics.to_csv(
    MODEL_DIR / "validation_segment_class_metrics.csv", index=False
)
test_segment_class_metrics.to_csv(
    MODEL_DIR / "test_segment_class_metrics.csv", index=False
)
validation_error_patterns.to_csv(
    MODEL_DIR / "validation_error_patterns.csv", index=False
)
test_error_patterns.to_csv(MODEL_DIR / "test_error_patterns.csv", index=False)
validation_segment_metrics.to_csv(
    MODEL_DIR / "validation_segment_metrics.csv", index=False
)
test_segment_metrics.to_csv(MODEL_DIR / "test_segment_metrics.csv", index=False)
if not feature_group_ablation.empty:
    feature_group_ablation.to_csv(
        MODEL_DIR / "temporal_feature_selection.csv", index=False
    )
if not temporal_feature_fold_scores.empty:
    temporal_feature_fold_scores.to_csv(
        MODEL_DIR / "temporal_feature_fold_scores.csv", index=False
    )

print("Model saved to:", model_path)
print("Metadata saved to:", metadata_path)
print("Model version:", metadata["model_version"])

sampling_weight_report.to_csv(
    MODEL_DIR / "training_sampling_weights.csv",
    index=True,
)
prior_correction_sweep.to_csv(
    MODEL_DIR / "prior_correction_power_sweep.csv",
    index=False,
)
training_population_prior.to_csv(
    MODEL_DIR / "natural_training_class_distribution.csv",
    index=False,
)


Model saved to: /home/ruthikreddy/Desktop/do2/do2_status_prediction/models_prac_v6_stable_bill_pattern/do2_catboost_multiclass.cbm
Metadata saved to: /home/ruthikreddy/Desktop/do2/do2_status_prediction/models_prac_v6_stable_bill_pattern/model_metadata_multiclass.json
Model version: do2-catboost-multiclass-20260831T060524Z


## What this run now proves

Use the validation report to judge the pre-2025 development model and the test
report to judge the deployment model refit through June 2025. The decisive
metrics are worst-month macro-F1, per-class recall, exact non-V accuracy, and
non-V recall—not overall accuracy alone. `created_month` remains diagnostic
only and is never included in `MODEL_FEATURE_COLUMNS`.


## Predict new DO2 rows

This function uses the deployment model refit through June 2025. Input rows
must contain ordinary entry-time DO2 columns and the same status-independent
SQL enrichment used during training. Supplied target or workflow-result
columns are ignored and never enter the model.


In [19]:
def predict_new_do2_rows(enriched_rows: pd.DataFrame) -> pd.DataFrame:
    """Return a safe I/MR/N/V/REVIEW decision for new entry-time rows.

    Only high-confidence V rows are auto-approved. Every unresolved case is
    REVIEW; uncertainty is never converted to V.
    """
    validate_prediction_columns(enriched_rows, REQUIRED_INPUT_COLUMNS)
    leakage_found = leakage_columns_present(enriched_rows.columns)
    model_input = enriched_rows.drop(
        columns=list(LEAKAGE_COLUMNS), errors="ignore"
    ).copy()

    feature_result = build_features(
        model_input,
        expected_columns=FEATURE_COLUMNS,
        expected_categorical=CATEGORICAL_COLUMNS,
        expected_text=TEXT_COLUMNS,
    )
    prepared = prepare_model_frame(feature_result.frame)
    probabilities = model.predict_proba(prepared[MODEL_FEATURE_COLUMNS])
    decisions, v_probability, argmax_status, argmax_probability = apply_safety_policy(
        probabilities, SAFETY_V_THRESHOLD, CLASS_THRESHOLDS
    )

    identity_columns = [
        column for column in (
            "id", "do2_no", "do2_item_no", "do2_date",
            "description", "do2_group", "do2_type", "transcription_type",
        )
        if column in enriched_rows.columns
    ]
    result = enriched_rows[identity_columns].copy()
    result["decision"] = decisions
    result["predicted_status"] = np.where(decisions == "REVIEW", pd.NA, decisions)
    result["model_argmax_status"] = argmax_status
    result["model_argmax_probability"] = argmax_probability
    result["probability_status_V"] = v_probability
    result["requires_manual_review"] = decisions == "REVIEW"
    result["auto_approved_as_v"] = decisions == "V"
    result["safety_v_threshold"] = SAFETY_V_THRESHOLD
    result["model_feature_group"] = SELECTED_FEATURE_GROUP
    result["feature_set_version"] = FEATURE_SET_VERSION
    for class_index, status_class in enumerate(STATUS_CLASSES):
        result[f"probability_status_{status_class}"] = probabilities[:, class_index]

    if leakage_found:
        print("Ignored leakage/target columns supplied with prediction input:", leakage_found)
    return result


# Example after obtaining entry-time enriched rows:
# new_predictions = predict_new_do2_rows(new_enriched_rows)
# display(new_predictions)
